# Clustering configs

## functions

In [ ]:
# #old
# def get_restults_per_cluster_old (json_path,  cluster, seeds, df_path, full= False,cluster_config = None, per_cluster = False, name = None, return_per_track = False, all_metrics = False ):
    
#     if full:
#         folder = os.path.join(json_path, "full_data", name)  if not all_metrics else os.path.join(json_path, "all_data_metrics", name)
#     elif all_metrics:
#         folder_metrics = os.path.join(json_path,"all_data_metrics")
#         cluster_config_folder = cluster_config if cluster_config is not None else ""
#         folder = os.path.join(folder_metrics, cluster_config_folder, name)
#         condition = ""
#     else:
#         #cluster_config = f"clusters{n_clusters}_layer{layer}"
#         folder =  os.path.join(json_path, cluster_config, f"{cluster}", name)
    
#     cluster_files, cluster_genres = get_cluster_files (cluster= cluster, df_path = df_path)
#     # if (per_cluster and  full) or (not full):
       
#     cluster_files = [file.replace("___", "/") for file in cluster_files]
#     #print(cluster_files)
#     scores = []
#     for seed in seeds:
#         condition = f"best-seed{seed}" if not all_metrics else f"epoch_best_seed_{seed}"
#         file_scores = [file for file in os.listdir(folder) if  file.startswith(condition)][0]
#         json_file = open(os.path.join(folder, file_scores))
        
#         json1_str = json_file.read()
#         json1_data = json.loads(json1_str)
#         json_data = {key[:-10]: value for key, value in json1_data.items()} #{key[:-10]: value for key, value in json1_data.items()}
#         df  = pd.DataFrame(json_data).T
        
#         #print(len(df))
#         #print(json_data)
#         if (full or all_metrics) and per_cluster:
#             ind = set(df.index)
#             cl = set(cluster_files)
#             #print(cl)
#             val_files_cluster = ind & cl
#             df_val_cluster = df.loc[list(val_files_cluster)]
#             df_val_cluster.index = df_val_cluster.index.astype(str) + "/track.npy"
#             #print(df_val_cluster.head(10))
#             #print(len(df_val_cluster))
#         else:
#             df_val_cluster = df.copy()
#         if seed == 0:
#             print(f"using file   {os.path.join(folder, file_scores)}")
#             print(f"number of files {df_val_cluster.shape}")
       
#         if return_per_track:
#             return df_val_cluster
#         df_percents = df_val_cluster*100

#         scores_mean = (df_percents .mean()).to_dict()
#         scores.append(scores_mean)

#     if not full:
#         genres = cluster_genres[cluster_genres.find("(")+1:cluster_genres.find(")")]
#         print(f"genres are {genres}")
#         return scores, genres
#     return scores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import matplotlib.patches as mpatches
import os
import json
import pandas as pd

def get_cluster_files (cluster, df_path):
    df_assignments = pd.read_csv(df_path)
    df_assignments_filtered =  df_assignments[ df_assignments["label"].str.contains(str(cluster))]
    cluster_files = df_assignments_filtered ["file"].values
    cluster_genres = [genres for genres in df_assignments_filtered ["label"].unique() if "(" in genres ][0]
    return cluster_files, cluster_genres



In [ ]:
import os
import json
import pandas as pd
import pandas as pd

def mean_scores(scores_list):
    """
    scores_list: List[Dict[str, float]]
    returns: Dict[str, float] with mean values
    """
    return pd.DataFrame(scores_list).mean().to_dict()
def std_scores(scores_list):
    """
    scores_list: List[Dict[str, float]]
    returns: Dict[str, float] with mean values
    """
    return pd.DataFrame(scores_list).std().to_dict()

def extract_mean_std(scores, extract_std = False, trial= None, decimals = 3, name = None):
    name = "" if name is None else f"{name}"
    suffix = f"{name}_baseline" if trial is None else f"{name}_model_{trial}"
    mean_dict = mean_scores(scores)
    if not extract_std:
        res = pd.DataFrame.from_dict(
            mean_dict,
            orient="index",
            columns=[suffix]
        ).T
    else:
        std_dict = std_scores(scores)

        # format as "95.2 +- 0.2"
        fmt = f"{{:.{decimals}f}}"
        combined = {
            k: f"{fmt.format(mean_dict[k])} +- {fmt.format(std_dict[k])}"
            for k in mean_dict.keys()
        }

        res = pd.DataFrame.from_dict(
            combined,
            orient="index",
            columns=[suffix]
        ).T
    return res

def remove_cemgil (df):
    wt_cemgil= [col for col in df.columns.values if "Cemgil" not in col]
    df= df[wt_cemgil]
    return df

def get_per_cluster_results(trials, seeds,json_path, cluster, name_full, cluster_config = None, 
                 df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv", trial_names = None, extrat_std = False, decimals = 3):
    df_list = []
    for trial in trials:
        name = trial_names.get(trial, "") if trial_names else ""
        scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
                    df_path=df_path, full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}{name}"
                
                )
        if len(seeds) >1:
            res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals)
         
        else:
            res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
            res = pd.DataFrame(res).T 
        # res = pd.DataFrame.from_dict(scores_cl2[0]  , orient="index", columns=[f"trial_{trial}"])
        # res = pd.DataFrame(res).T 
        
        
        df_list.append(res)

    scores_base = get_restults_per_cluster(
                json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
                df_path=df_path, full=True, filter_files=True, name = name_full
            )
    
    
    if len(seeds) >1:
        res_base = extract_mean_std(scores_base, trial = None, extract_std=extrat_std, decimals = decimals)
    else:
        res_base = pd.DataFrame.from_dict(scores_base[0]  , orient="index", columns=[f"baseline"])
        res_base = pd.DataFrame(res_base).T
    df_list.append(res_base)
    df_total = pd.concat(df_list, axis=0)
    df_total = remove_cemgil(df_total)
    # wt_cemgil= [col for col in df_total.columns.values if "Cemgil" not in col]
    # df_total= df_total[wt_cemgil]
    return df_total 


def get_baseliine_diff(df_total, num_trials):
    df_baseline = df_total.loc["baseline"]
    diff_vs_baseline = df_total[:num_trials].subtract(df_total.loc["baseline"])
    diff_vs_baseline
    diff_vs_baseline = pd.concat([diff_vs_baseline, df_baseline.to_frame().T], axis=0)
    return diff_vs_baseline


def get_latex_row(df, remove_cemgil = True):
    if remove_cemgil:
        wt_cemgil= [col for col in df.columns.values if "Cemgil" not in col]
        df= df[wt_cemgil]
    rows = [
    " & ".join(f"{x:.3f}" if isinstance(x, (int, float)) else str(x) for x in row)
    for row in df.values
]
    return rows

In [53]:
import re
import numpy as np
def _extract_mean(value):
    """Extract numeric mean from strings like '96.643 +- 0.114', '96.643 ± 0.114', '96.643 \\pm 0.114'."""
    s = str(value)
    # strip math and whitespace
    s = s.replace("$", "").strip()
    # normalise all kinds of +- / ± / \pm to '+-'
    s = s.replace(r"\pm", "+-").replace("±", "+-")
    if "+-" in s:
        mean_part = s.split("+-")[0].strip()
    else:
        # fall back: first token
        mean_part = s.split()[0].strip()
    try:
        return float(mean_part)
    except ValueError:
        return np.nan

In [58]:
import re
import numpy as np

_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def _extract_mean(value):
    """
    Robustly extract the first numeric value from a cell like:
    '96.643 +- 0.114', '96.643 ± 0.114', '96.643 \\pm 0.114',
    '\\mathbf{96.64} \\pm 0.11', 'clusters4_layer12 92.81 ± 0.11'
    """
    if value is None:
        return np.nan
    s = str(value).replace("$", "").strip()

    m = _num_re.search(s)
    return float(m.group(0)) if m else np.nan


In [59]:
def df_to_latex_rows(
    df,
    rename_idx=None,
    bold_index=True,
    pm=False,
    bold_difference=False,
    bold_baseline_if_best=False
):
    rows = []

    # ---------- locate baseline ----------
    baseline_idx = None
    for idx in df.index:
        if "baseline" in str(idx).lower():
            baseline_idx = idx
            break

    baseline_means = {}
    if baseline_idx is not None:
        for col in df.columns:
            baseline_means[col] = _extract_mean(df.loc[baseline_idx, col])

    # ---------- if we bold baseline when best, precompute ----------
    baseline_best_cols = set()
    if bold_baseline_if_best and baseline_idx is not None:
        for col in df.columns:
            others = [
                _extract_mean(df.loc[i, col])
                for i in df.index
                if i != baseline_idx
            ]
            if len(others) > 0:
                if baseline_means[col] >= np.nanmax(others):
                    baseline_best_cols.add(col)

    # ---------- build rows ----------
    for i, (idx, row) in enumerate(df.iterrows()):
        label = str(rename_idx[i]) if rename_idx is not None else str(idx)
        if bold_index:
            label = rf"\textbf{{{label}}}"

        is_baseline_row = "baseline" in str(idx).lower()
        cell_strs = []

        for col, v in row.items():
            s = str(v)
            s = s.replace("+-", r"\pm")
            if pm:
                s = s.replace("±", r"\pm")
            else:
                s = s.replace("±", r"\pm")

            # -------- bold logic --------
            bold_mean_here = False

            if bold_difference and not is_baseline_row and col in baseline_means:
                mean_val = _extract_mean(v)
                base_val = baseline_means[col]
                if not np.isnan(mean_val) and mean_val > base_val:
                    bold_mean_here = True

            if bold_baseline_if_best and is_baseline_row and col in baseline_best_cols:
                bold_mean_here = True

            if bold_mean_here:
                pm_pos = s.find(r"\pm")
                if pm_pos != -1:
                    mean_part = s[:pm_pos].strip()
                    rest = s[pm_pos:].lstrip()
                    s = rf"\mathbf{{{mean_part}}} {rest}"
                else:
                    s = rf"\mathbf{{{s}}}"

            cell_strs.append(f"${s}$")

        row_str = " & ".join([label] + cell_strs) + r" \\"
        rows.append(row_str)

    return rows

In [67]:
def df_to_latex_rows(
    df,
    rename_idx=None,
    bold_index=True,
    pm=False,
    bold_difference=False,
    bold_baseline_if_best=False,
    bold_max_per_col=False,          # NEW
):
    rows = []

    # ---------- locate baseline ----------
    baseline_idx = None
    for idx in df.index:
        if "baseline" in str(idx).lower():
            baseline_idx = idx
            break

    baseline_means = {}
    if baseline_idx is not None:
        for col in df.columns:
            baseline_means[col] = _extract_mean(df.loc[baseline_idx, col])

    # ---------- baseline best cols (old behavior) ----------
    baseline_best_cols = set()
    if bold_baseline_if_best and baseline_idx is not None:
        for col in df.columns:
            others = [
                _extract_mean(df.loc[i, col])
                for i in df.index
                if i != baseline_idx
            ]
            if len(others) > 0:
                if baseline_means[col] >= np.nanmax(others):
                    baseline_best_cols.add(col)

    # ---------- NEW: global max (per column) ----------
    max_cells = set()  # store (row_idx, col) pairs to bold
    if bold_max_per_col:
        for col in df.columns:
            means = {i: _extract_mean(df.loc[i, col]) for i in df.index}
            print(col)
            print(means)
            max_val = np.nanmax(list(means.values()))
            for i, mv in means.items():
                if not np.isnan(mv) and mv == max_val:
                    max_cells.add((i, col))
    
    # ---------- build rows ----------
    for i, (idx, row) in enumerate(df.iterrows()):
        label = str(rename_idx[i]) if rename_idx is not None else str(idx)
        if bold_index:
            label = rf"\textbf{{{label}}}"

        is_baseline_row = "baseline" in str(idx).lower()
        cell_strs = []

        for col, v in row.items():
            s = str(v).replace("+-", r"\pm")
            s = s.replace("±", r"\pm")  # always normalize

            bold_mean_here = False

            # NEW: bold global max per column
            if bold_max_per_col and (idx, col) in max_cells:
                bold_mean_here = True

            # old logic can still apply in addition (or you can make it elif)
            if bold_difference and not is_baseline_row and col in baseline_means:
                mean_val = _extract_mean(v)
                base_val = baseline_means[col]
                if not np.isnan(mean_val) and mean_val > base_val:
                    bold_mean_here = True

            if bold_baseline_if_best and is_baseline_row and col in baseline_best_cols:
                bold_mean_here = True

            if bold_mean_here:
                pm_pos = s.find(r"\pm")
                if pm_pos != -1:
                    mean_part = s[:pm_pos].strip()
                    rest = s[pm_pos:].lstrip()
                    s = rf"\mathbf{{{mean_part}}} {rest}"
                else:
                    s = rf"\mathbf{{{s}}}"

            cell_strs.append(f"${s}$")

        rows.append(" & ".join([label] + cell_strs) + r" \\")
    return rows


In [ ]:
# def df_to_latex_rows(df, rename_idx=None, bold_index = True, pm = False, bold_difference = False):
#     """
#     rename_idx: list of new names in the SAME order as df rows
#                 e.g., ["Cluster Model", "Baseline"]
#     """
#     rows = []
    
#     for i, (idx, row) in enumerate(df.iterrows()):
#         label = str(rename_idx[i]) if rename_idx is not None else str(df.index[i])
        
#         if bold_index:
#             label = rf"\textbf{{{label}}}"
#         # choose label from list if provided
#         # if rename_idx is not None:
#         #     label = str(rename_idx[i])
#         # else:
#         #     label = str(idx)
        
#         vals = [str(v).replace("+-", r"\pm") for v in row]
#         if pm:
#             vals = [str(v).replace("±", r"\pm") for v in row]
#         vals = [f"${v}$" for v in vals]

#         row_str = " & ".join([label] + vals) + r" \\"
#         rows.append(row_str)

#     return rows

In [ ]:
import pandas as pd
import os
import json
import numpy as np
# create df of results for the baseline model trained on the 
# full data with a corresponding seed
def construct_df_full_data (json_path,  seed, rename_columns = None, names = None, best = True, subfolder = ""):
    # if names:
    #     name = names[0]
    #     subfolder = "full_data"
    # else:
    #     name = ""
    #     subfolder = "full_data_intermediate_checkpoints"
    name = names[0] if names else ""
    folder = os.path.join(json_path, subfolder, name)  # 
    
    prefix = f"best-seed{seed}" if best else f"seed_{seed}"
    if seed == 0:
        print(folder)
        print(prefix)
    file_scores = [file for file in os.listdir(folder) if prefix in file][0]
    #print(file_scores)
    json_file = open(os.path.join(folder, file_scores))
    json1_str = json_file.read()
    json1_data = json.loads(json1_str)
    clsuter_keys = set(json1_data.keys())
    if seed == 0:
        print(os.path.join(folder, file_scores))
    #print(len(clsuter_keys))
    df  = pd.DataFrame(json1_data).T
    if rename_columns is not None:
       mapping = { col: f"{col}_{rename_columns}_{seed}" for col in df.columns}
       df = df.rename(columns=mapping)
    return df

# cerate df of results for the clustering configutation for one cluster
def construct_df_clsuter_data(json_path, seed, cluster_config, cluster_number,rename_columns= None, names = None, all_metrics= False):
    if names:
        name = names[cluster_number]
    else:
        name = ""
    folder = os.path.join(json_path, cluster_config, f"{cluster_number}", name)
    # if seed == 0:
    #     print(folder)
    file_scores = [file for file in os.listdir(folder) if  file.startswith(f"best-seed{seed}")][0]   # f"best_checkpoint-seed{seed}" in file best_checkpoint-seed

    json_file = open(os.path.join(folder, file_scores))
    if seed == 0:
        print(os.path.join(folder, file_scores))
        #print(df.shape)
    json1_str = json_file.read()
    json1_data = json.loads(json1_str)
    clsuter_keys = set(json1_data.keys())
    #print(len(clsuter_keys))
    df  = pd.DataFrame(json1_data).T
    if rename_columns is not None:
       mapping = { col: f"{col}_{rename_columns}_{seed}" for col in df.columns}
       df = df.rename(columns=mapping)
    return df

def get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = None, filter_files = False, name = None, return_per_track = False, subfolder = "", cluster_model= None ):
    cluster_model = cluster_model if cluster_model is not None else cluster
    if cluster_config:
        folder =   os.path.join(json_path, subfolder,cluster_config, f"{cluster_model}", name)
    else: 
        folder =  os.path.join(json_path, subfolder, name)
    
    cluster_files, cluster_genres = get_cluster_files (cluster= cluster, df_path = df_path)
    # if (per_cluster and  full) or (not full):
       
    cluster_files = [file.replace("___", "/") for file in cluster_files]
    #print(cluster_files)
    scores = []
    for seed in seeds:
        #condition = f"best-seed{seed}" if not all_metrics else f"epoch_best_seed_{seed}"
        #print(folder)
        #file_scores = [file for file in os.listdir(folder) if  (f"seed{str(seed)}" or f"seed_{str(seed)}") in file and not file.startswith("0")][0]
        file_scores = [
            file for file in os.listdir(folder)
            if (
                f"seed{seed}" in file
                or f"seed_{seed}" in file
            )
            and not file.startswith("0")
        ][0]
        json_file = open(os.path.join(folder, file_scores))
        
        json1_str = json_file.read()
        json1_data = json.loads(json1_str)
        json_data = {key[:-10]: value for key, value in json1_data.items()} #{key[:-10]: value for key, value in json1_data.items()}
        df  = pd.DataFrame(json_data).T
        
        #print(len(df))
        #print(json_data)
        if filter_files:
            ind = set(df.index)
            cl = set(cluster_files)
            #print(cl)
            val_files_cluster = ind & cl
            df_val_cluster = df.loc[list(val_files_cluster)]
            df_val_cluster.index = df_val_cluster.index.astype(str) + "/track.npy"
            #print(df_val_cluster.head(10))
            #print(len(df_val_cluster))
        else:
            df_val_cluster = df.copy()
        if seed == 0:
            print(f"using file   {os.path.join(folder, file_scores)}")
            print(f"number of files {df_val_cluster.shape}")
       
        if return_per_track:
            return df_val_cluster
        df_percents = df_val_cluster*100

        scores_mean = (df_percents .mean()).to_dict()
        scores.append(scores_mean)

    if not full:
        genres = cluster_genres[cluster_genres.find("(")+1:cluster_genres.find(")")]
        print(f"genres are {genres}")
        return scores, genres
    return scores
# join results for all clusters in the given clustering configuration and join then in final df
def construct_total_df (json_path, seed, cluster_config, cluster_numbers, rename_columns= None, names = None, skip_cluster = [], df_path = None, all_metrics = False):
    df_list = []
    #print(skip_cluster)
    for cluster_number in cluster_numbers:
        if cluster_number in skip_cluster:
            print(f"skipping cluster {cluster_number}")
            df_cluster  = get_restults_per_cluster(
                json_path, cluster=cluster_number, seeds=[seed], cluster_config=None,
                df_path=df_path, full=True, per_cluster=True, name = names[0], return_per_track=True, all_metrics= all_metrics
            )
            #col_down = [col for col in df_cluster.columns.values if "F-measure_downbeat" in col][0]
            #print(f"total sum for cluster {cluster_number} skipped is {df_cluster[col_down].sum()}")
            # res_base = pd.DataFrame.from_dict(scores_cl_base[0]  , orient="index", columns=[f"baseline"])
            # df_cluster = pd.DataFrame(res_base)
            
            
        else:
            
            df_cluster = construct_df_clsuter_data(json_path, seed, cluster_config, cluster_number, rename_columns= rename_columns, names = names, all_metrics= all_metrics)
            #col_down = [col for col in df_cluster.columns.values if "F-measure_downbeat" in col][0]
           # print(f"total sum for cluster {cluster_number} is {df_cluster[col_down].sum()}")
        #print(f"cluster len is {len(df_cluster)}")
        # if seed == 0:
        #     print(df_cluster.shape)
        df_list.append (df_cluster)
    df_total = pd.concat(df_list, axis=0)
    df = df_total.groupby(df_total.index).mean()
    # sanity check on number of validation files (should be 556)
    #assert len(df) == 556
    #print(f"Final size of df is {len(df)}")
    return df

# get dict of differences between the baseline model and configuration model
# returns dict with keys that are beat tracking metrics
def get_performance_differences(df_total, df_full):
    means_total = (df_total.mean()* 100).to_dict()
    means_full =(df_full.mean()*  100).to_dict()
    diff_dict = {list(means_total.keys())[i]: means_total[list(means_total.keys())[i]] - means_full[list(means_full.keys())[i]] for i in range(len(means_full))}
    return diff_dict

# utility function to rename dictionary keys
def rename_keys_dict(d):
    dict_new = {}
    for key in d.keys():
        new_key = "_".join(key.split("_")[:-1])
        dict_new [new_key] = d[key]
    return dict_new

# get differences in performance for a particular configuration for all seeds
def get_total_results (seeds, json_val_path,layer, clusters, ):
    df_list = []
    cluster_numbers = np.arange(clusters) +1
    cluster_config = f"clusters{clusters}_layer{layer}"
    for seed in seeds:
        df_total = construct_total_df (json_val_path, seed=seed, cluster_config=cluster_config, cluster_numbers=cluster_numbers)
        df_full = construct_df_full_data (json_val_path, seed=seed, rename_columns="baseline")
        diff_dict = get_performance_differences (df_total, df_full)
        diff_dict_renamed = rename_keys_dict(diff_dict)
        #print(diff_dict_renamed)
        df = pd.DataFrame.from_dict(diff_dict_renamed, orient="index", columns=[f"Value_{seed}"])
        #df.index.name = "Metric"
        df_list.append(df)
    df_result =  pd.concat(df_list, axis=1)
    return df_result

In [ ]:
def construct_total_df_new (json_path, seed, cluster_config, cluster_numbers, rename_columns= None, names = None, skip_cluster = [], df_path = None, subfolders= ["", ""]):
    df_list = []
    #print(skip_cluster)
    for cluster_number in cluster_numbers:
        if cluster_number in skip_cluster:
            if seed ==0:
                print(f"skipping cluster {cluster_number}")
            df_cluster  = get_restults_per_cluster(
                json_path, cluster=cluster_number, seeds=[seed], cluster_config=None,
                df_path=df_path, full=True, filter_files=True, name = names[0], return_per_track=True, subfolder= subfolders[0]
            )
            #col_down = [col for col in df_cluster.columns.values if "F-measure_downbeat" in col][0]
            #print(f"total sum for cluster {cluster_number} skipped is {df_cluster[col_down].sum()}")
            # res_base = pd.DataFrame.from_dict(scores_cl_base[0]  , orient="index", columns=[f"baseline"])
            # df_cluster = pd.DataFrame(res_base)
            
            
        else:
            df_cluster  = get_restults_per_cluster(
                json_path, cluster=cluster_number, seeds=[seed], cluster_config=cluster_config,
                df_path=df_path, full=False, filter_files=True, name = names[cluster_number], return_per_track=True, subfolder= subfolders[1]
            )
           
            #col_down = [col for col in df_cluster.columns.values if "F-measure_downbeat" in col][0]
           # print(f"total sum for cluster {cluster_number} is {df_cluster[col_down].sum()}")
        #print(f"cluster len is {len(df_cluster)}")
            if seed == 0:
                print(df_cluster.shape)
        df_list.append (df_cluster)
    df_total = pd.concat(df_list, axis=0)
    df = df_total.groupby(df_total.index).mean()
    # sanity check on number of validation files (should be 556)
    #assert len(df) == 556
    #print(f"Final size of df is {len(df)}")
    return df

In [ ]:
def get_total_results_new(seeds, json_val_path,layer = None, clusters = None, full = False, rename_columns = None, names = None, skip_cluster = [], df_path = None, decimals = 2, best = True,subfolders= ["", ""]):
    df_list = []
    for seed in seeds:
        if full:
            df_total = construct_df_full_data (json_val_path, seed=seed, rename_columns=rename_columns, names = names, best= best, subfolder= subfolders[0])
            col_down = [col for col in df_total.columns.values if "F-measure_downbeat" in col][0]
            sum_full = df_total[col_down].sum() 
            #print(f"total sum is {sum_full}")   
            if seed == 0:
                print(len(df_total))
        else:
            cluster_numbers = np.arange(clusters) +1
            cluster_config = f"clusters{clusters}_layer{layer}"
            df_total = construct_total_df_new (json_val_path, seed=seed, cluster_config=cluster_config, cluster_numbers=cluster_numbers, 
                                           rename_columns=rename_columns, names = names, skip_cluster = skip_cluster, df_path = df_path, subfolders=subfolders)
            #print(f"the total cluster dif is {len(df_total)}")
        means_total = (df_total.mean()* 100).to_dict()
        means_total_renamed =means_total.copy()# rename_keys_dict(means_total)
        #print(diff_dict_renamed)
        df = pd.DataFrame.from_dict(means_total_renamed  , orient="index", columns=[f"configuration_{seed}"])
        #df.index.name = "Metric"
        df_list.append(df)
    df_result =  pd.concat(df_list, axis=1)
    df_seeds_mean = df_result.mean(axis = 1)
    df_seeds_std = df_result.std(axis = 1)
    fmt = f"{{:.{decimals}f}}"
    def format_numeric(x):
        return fmt.format(x)
    #df_final = df_seeds_mean.round(decimals).astype(str) + " ± " + df_seeds_std.round(decimals).astype(str)
    df_final = (
    df_seeds_mean.apply(format_numeric)
    + " ± "
    + df_seeds_std.apply(format_numeric)
    )
    return df_seeds_mean, df_seeds_std, df_final

## results

In [ ]:
seeds = [0,1,2]
json_val_path =  "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , full = True ) #rename_columns="baseline"
# df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 6, clusters = 2)
# df_seeds_mean, df_seeds_std, df_final_clusters4_layer12 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 12, clusters = 4)
# df_seeds_mean, df_seeds_std, df_final_clusters3_layer113= get_total_results_new(seeds =seeds , json_val_path = json_val_path , layer = 113, clusters = 3)
configs = {
    # "clusters2_layer6": df_final_clusters2_layer6,
    # "clusters4_layer12": df_final_clusters4_layer12,
    
    # "clusters3_raw" : df_final_clusters3_layer113,
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [ ]:
seeds = [0,1,2]
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = "json_val_scores", full = True ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results_new(seeds = seeds, json_val_path = "json_val_scores", layer = 6, clusters = 2)
df_seeds_mean, df_seeds_std, df_final_clusters4_layer12 = get_total_results_new(seeds = seeds, json_val_path = "json_val_scores", layer = 12, clusters = 4)
df_seeds_mean, df_seeds_std, df_final_clusters3_layer113= get_total_results_new(seeds =seeds , json_val_path = "json_val_scores", layer = 113, clusters = 3)
configs = {
    "clusters2_layer6": df_final_clusters2_layer6,
    "clusters4_layer12": df_final_clusters4_layer12,
    
    "clusters3_raw" : df_final_clusters3_layer113,
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [ ]:
seeds = [0,1,2]
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = "json_val_scores", full = True ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results(seeds = seeds, json_val_path = "json_val_scores", layer = 6, clusters = 2)
df_seeds_mean, df_seeds_std, df_final_clusters4_layer12 = get_total_results(seeds = seeds, json_val_path = "json_val_scores", layer = 12, clusters = 4)
df_seeds_mean, df_seeds_std, df_final_clusters3_layer113= get_total_results(seeds =seeds , json_val_path = "json_val_scores", layer = 113, clusters = 3)
configs = {
    "clusters2_layer6": df_final_clusters2_layer6,
    "clusters4_layer12": df_final_clusters4_layer12,
    
    "clusters3_raw" : df_final_clusters3_layer113,
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [ ]:
def extract_mean(series):
    return series.str.split("±").str[0].astype(float)
df_means_only = final_df.apply(extract_mean, axis=1)
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

In [ ]:
df_seeds_mean, df_seeds_std, df_final = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", full = True ) #rename_columns="baseline"
df_final

In [ ]:
df_seeds_mean, df_seeds_std, df_final = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 6, clusters = 2)
df_final

In [ ]:
df_seeds_mean, df_seeds_std, df_final= get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 12, clusters = 4)
df_final

In [ ]:
df_seeds_mean, df_seeds_std, df_final= get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 113, clusters = 3)
df_final

In [ ]:
df2 = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 6, clusters = 2)
df2

In [ ]:
# using df with per track cluster assignments, return only tracks
# that belong to the chosen cluster
def get_cluster_files (cluster, df_path):
    df_assignments = pd.read_csv(df_path)
    df_assignments_filtered =  df_assignments[ df_assignments["label"].str.contains(str(cluster))]
    cluster_files = df_assignments_filtered ["file"].values
    cluster_genres = [genres for genres in df_assignments_filtered ["label"].unique() if "(" in genres ][0]
    return cluster_files, cluster_genres

#compute scores only for tracks from a given cluster
# when per_cluster set to False, we compute scores for all tracks
def old_get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False, layer = None, n_clusters = None, per_cluster = True ):
    if full:
        folder = os.path.join(json_path, "full_data")
    else:
        cluster_config = f"clusters{n_clusters}_layer{layer}"
        folder =  os.path.join(json_path, cluster_config, f"{cluster}")
    cluster_files, cluster_genres = get_cluster_files (cluster= cluster, df_path = df_path)
    print(f"genres are {cluster_genres}")
    cluster_files = [file.replace("___", "/") for file in cluster_files]
    #print(cluster_files)
    scores = []
    for seed in seeds:
        file_scores = [file for file in os.listdir(folder) if  file.startswith(f"best-seed{seed}")][0]
        json_file = open(os.path.join(folder, file_scores))
        json1_str = json_file.read()
        json1_data = json.loads(json1_str)
        json_data =json1_data.copy() #{key[:-10]: value for key, value in json1_data.items()}  # !!!!!!!
        df  = pd.DataFrame(json_data).T
        if full and per_cluster:
            ind = set(df.index)
            cl = set(cluster_files)
            val_files_cluster = ind & cl
            df_val_cluster = df.loc[list(val_files_cluster)]
        else:
            df_val_cluster = df.copy()
        scores_mean = (df_val_cluster.mean()*100).to_dict()
        scores.append(scores_mean)
    print(len(df_val_cluster))
    return scores

In [ ]:
df2 = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 6, clusters = 2)
clusters2_layer6 = df2.mean(axis = 1).to_dict()
clusters2_layer6 = {key.split("_clusters")[0]: value for key, value in clusters2_layer6.items()}
clusters2_layer6 
#df2
df_res2 = pd.DataFrame.from_dict(clusters2_layer6 , orient="index", columns=[f"Value_{1}"])
df_res2

In [ ]:
df4 = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 12, clusters = 4)
df4.mean(axis = 1)


In [ ]:
df3 = get_total_results(seeds = [0,1, 2], json_val_path = "json_val_scores", layer = 113, clusters = 3)
df3.mean(axis = 1)


In [ ]:
df2 = get_total_results(seeds = [0,1,2], json_val_path = "json_val_scores", layer = 6, clusters = 2)
df2.mean(axis = 1)


In [ ]:

json_path = "json_val_scores"
scores = get_restults_per_cluster (json_path,  cluster = 4, seeds = [0,1,2], df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", full = True, per_cluster=False )
df = pd.DataFrame(scores)

summary = pd.DataFrame({
    "mean": df.mean(),
    "std": df.std()
})
summary

In [ ]:
json_path = "json_val_scores"
layer = 12
cluster_numbers = [1,2,3,4]
df_list = []
seed = 0
cluster_config = "clusters4_layer12"
for cluster_number in cluster_numbers:
    df_cluster = construct_df_clsuter_data(json_path, seed, cluster_config, cluster_number, rename_columns= cluster_config)
    df_list.append (df_cluster)
df_total = pd.concat(df_list, axis=0)
df = df_total.groupby(df_total.index).mean()
# sanity check on number of validation files (should be 556)
print(f"Final size of df is {len(df)}")
df.head()

In [ ]:
cluster_files, cluster_genres = get_cluster_files (cluster= 1, config="clusters4_layer12" )
len(cluster_files)
cluster_files = [file.replace("___", "/") for file in cluster_files]
cluster_files

In [ ]:
json_path = "json_val_scores"
seed = 0
folder = os.path.join(json_path, "full_data")
file_scores = [file for file in os.listdir(folder) if f"best-seed{seed}" in file][0]

json_file = open(os.path.join(folder, file_scores))
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json_data = {key[:-10]: value for key, value in json1_data.items()}
clsuter_keys = set(json_data.keys())
print(len(clsuter_keys))
df  = pd.DataFrame(json_data).T
ind = set(df.index)
cl = set(cluster_files)
val_files_cluster = ind & cl
df_val_cluster = df.loc[list(val_files_cluster)]
df_val_cluster.head()

In [ ]:
json_path = "json_val_scores"
seed = 1
folder = os.path.join(json_path, "full_data")
file_scores = [file for file in os.listdir(folder) if f"best-seed{seed}" in file][0]

json_file = open(os.path.join(folder, file_scores))
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json_data = {key[:-10]: value for key, value in json1_data.items()}
clsuter_keys = set(json_data.keys())
print(len(clsuter_keys))
df  = pd.DataFrame(json_data).T
ind = set(df.index)
cl = set(cluster_files)
val_files_cluster = ind & cl
df_val_cluster1 = df.loc[list(val_files_cluster)]
(df_val_cluster1.mean()*100).to_dict()

In [ ]:
import matplotlib.pyplot as plt
def gete_f1_av(scores):
    df = pd.DataFrame(scores)
    summary = pd.DataFrame({
        "mean": df.mean(),
        "std": df.std()
    })
    return summary.loc["F-measure_beat", "mean"]
def bar_plot_per_cluster (df_path, json_path, clusters, seeds, cluster_config ):
    full_scores =  {}
    config_scores = {}
    for cluster in clusters:
        scores_full = get_restults_per_cluster (json_path,  cluster =cluster, seeds = seeds, df_path=df_path, full = True, per_cluster=True )
        scores_config, cluster_genres = get_restults_per_cluster (json_path,  cluster =cluster, seeds = seeds, df_path=df_path, full = False, cluster_config=cluster_config )
        full_scores[cluster] = gete_f1_av(scores_full)
        config_scores[cluster] = gete_f1_av(scores_config)
    fig, ax = plt.subplots(figsize=(10,5))
        #ax.barplot()
        
    ax.bar(clusters, list(full_scores.values()), align='center')


## bar plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import matplotlib.patches as mpatches

def clean_label(label, width=12):
    # Replace + by comma
    label = label.replace(" +", ",")
    # Remove accidental double spaces
    label = " ".join(label.split())
    # Wrap to multiple lines
    return "\n".join(textwrap.wrap(label, width=width))

def bar_plot_per_cluster(df_path, json_path, clusters, seeds, cluster_config):
    full_scores = {}
    config_scores = {}
    cluster_labels = []   # for x-tick labels (genres per cluster)

    for cluster in clusters:
        scores_full = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds,
            df_path=df_path, full=True, per_cluster=True
        )
        scores_config, cluster_genres = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds,
            df_path=df_path, full=False, cluster_config=cluster_config
        )

        full_scores[cluster] = gete_f1_av(scores_full)
        config_scores[cluster] = gete_f1_av(scores_config)

        # assuming cluster_genres is a list of strings; adjust if needed
        cluster_labels.append(cluster_genres)

    # --- plotting ---
    clusters = list(clusters)
    x = np.arange(len(clusters))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 5))

    # one color per cluster
    colors = plt.cm.tab10(np.arange(len(clusters)) % 10)

    full_vals = [full_scores[c] for c in clusters]
    config_vals = [config_scores[c] for c in clusters]

    for i, c in enumerate(clusters):
        # full scores: more opaque
        ax.bar(
            x[i] - width / 2,
            full_vals[i],
            width,
            color=colors[i],
            alpha=0.4,
            label="Full data" if i == 0 else ""
        )
        # config scores: same color, less opaque
        ax.bar(
            x[i] + width / 2,
            config_vals[i],
            width,
            color=colors[i],
            alpha=0.9,
            label="Cluster-config" if i == 0 else ""
        )
    cluster_labels_clean = [clean_label(lbl) for lbl in cluster_labels]
    ax.set_xticks(x)
    ax.set_xticklabels(cluster_labels_clean,  ha="center")
    ax.set_ylabel("F1 score (beat)")   # or whatever your metric is
    ax.set_xlabel("Clusters (main genres)")
    #ax.set_title("Beat F1 per cluster: full vs cluster-specific config")
    full_patch = mpatches.Patch(color="gray", alpha=0.4, label="Full data")
    config_patch = mpatches.Patch(color="gray", alpha=0.9, label="Cluster-config")

    ax.legend(handles=[full_patch, config_patch])
    #ax.legend()
    ax.set_ylim(75, 100)
    fig.tight_layout()

    return config_vals, full_vals

In [ ]:
bar_plot_per_cluster (df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", json_path =  "json_val_scores",
                       clusters = [1,2,3,4], seeds =[0,1,2] , cluster_config= "clusters4_layer12")

In [ ]:
df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
json_path =  "json_val_scores"
clusters = [1,2,3,4]
seeds =[0,1,2]
cluster_config= "clusters4_layer12"
full_scores = {}
config_scores = {}
cluster_labels = []   # for x-tick labels (genres per cluster)

for cluster in clusters:
    scores_full = get_restults_per_cluster(
        json_path, cluster=cluster, seeds=seeds,
        df_path=df_path, full=True, per_cluster=True
    )
    scores_config, cluster_genres = get_restults_per_cluster(
        json_path, cluster=cluster, seeds=seeds,
        df_path=df_path, full=False, cluster_config=cluster_config
    )

    full_scores[cluster] = gete_f1_av(scores_full)
    config_scores[cluster] = gete_f1_av(scores_config)

    # assuming cluster_genres is a list of strings; adjust if needed
    cluster_labels.append(cluster_genres)


In [ ]:
bar_plot_per_cluster (df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", json_path =  "json_val_scores",
                       clusters = [1,2,3,4], seeds =[0,1,2] , cluster_config= "clusters4_layer12")

In [ ]:
bar_plot_per_cluster (df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv", json_path =  "json_val_scores",
                       clusters = [1,2], seeds =[0,1,2] , cluster_config= "clusters2_layer6")

In [ ]:
bar_plot_per_cluster (df_path = "data_cluster_assignments/df_raw_113_3clusters_new.csv", json_path =  "json_val_scores",
                       clusters = [1,2, 3], seeds =[0,1,2] , cluster_config= "clusters3_layer113")

In [ ]:

json_path = "json_val_scores"
scores = get_restults_per_cluster (json_path,  cluster = 4, seeds = [0,1,2], df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", full = True, per_cluster=True )
df = pd.DataFrame(scores)

summary = pd.DataFrame({
    "mean": df.mean(),
    "std": df.std()
})
summary.loc["F-measure_beat", "mean"]

In [ ]:
scores = get_restults_per_cluster (json_path,  cluster = 4, seeds = [0,1,2], df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", full =False,cluster_config="clusters4_layer12")
df = pd.DataFrame(scores)

summary = pd.DataFrame({
    "mean": df.mean(),
    "std": df.std()
})
summary

In [ ]:
len(scores)

In [ ]:
cluster_config = "clusters4_layer12"
cluster_number = 1
folder = os.path.join(json_path, cluster_config, f"{cluster_number}")
print(folder)
file_scores = [file for file in os.listdir(folder) if  file.startswith(f"best-seed{seed}")][0]   # f"best_checkpoint-seed{seed}" in file best_checkpoint-seed

json_file = open(os.path.join(folder, file_scores))
json1_str = json_file.read()
json1_data = json.loads(json1_str)
clsuter_keys = set(json1_data.keys())
print(len(clsuter_keys))
df  = pd.DataFrame(json1_data).T
df.head()

In [ ]:
def get_total_results (seeds, json_val_path, cluster_config, cluster_numbers):
    df_list = []
    for seed in seeds:
        df_total = construct_total_df (json_val_path, seed=seed, cluster_config=cluster_config, cluster_numbers=cluster_numbers)
        df_full = construct_df_full_data (json_val_path, seed=seed, rename_columns="baseline")
        diff_dict = get_performance_differences (df_total, df_full)
        diff_dict_renamed = rename_keys_dict(diff_dict)
        #print(diff_dict_renamed)
        df = pd.DataFrame.from_dict(diff_dict_renamed, orient="index", columns=[f"Value_{seed}"])
        #df.index.name = "Metric"
        df_list.append(df)
    df_result =  pd.concat(df_list, axis=1)
    return df_result

In [ ]:
import json
import os
json_val_path = "json_val_scores/clusters4_layer12"
keys = {}
for i in range (1,5):
    folder = os.path.join(json_val_path, str(i))
    file_scores = [file for file in os.listdir(folder) if "best_checkpoint" in file][0]

    json_file = open(os.path.join(folder, file_scores))
    json1_str = json_file.read()
    json1_data = json.loads(json1_str)
    clsuter_keys = set(json1_data.keys())
    keys[i] = clsuter_keys
print(keys)

In [ ]:
df1 = construct_df_clsuter_data(json_val_path, seed=0,cluster_config = "clusters2_layer6", cluster_number= 1,rename_columns= "clusters2_layer6")
df2 = construct_df_clsuter_data(json_val_path, seed=0,cluster_config = "clusters2_layer6", cluster_number= 2,rename_columns= "clusters2_layer6")
df_total = pd.concat([df1, df2], axis=0)
cols_f = [col for col in df_total.columns if "F-measure" in col]
df_total_f1 = df_total[cols_f]
df_total_f1 
means_total = (df_total.mean()* 100).to_dict()
means_total

In [ ]:
df_full = construct_df_full_data (json_val_path, seed=0, rename_columns="baseline")
cols_f = [col for col in df_full.columns if "F-measure" in col]
df_fulll_f1 = df_full[cols_f]
means_full =(df_full.mean()*  100).to_dict()
means_full

In [ ]:
diff_dict = {list(means_total.keys())[i]: means_total[list(means_total.keys())[i]] - means_full[list(means_full.keys())[i]] for i in range(len(means_full))}
diff_dict

In [ ]:
df_total = construct_total_df (json_val_path, seed=0, cluster_config="clusters2_layer6", cluster_numbers=[1,2])
df_full = construct_df_full_data (json_val_path, seed=0, rename_columns="baseline")
diff_dict = get_performance_differences (df_total, df_full)
diff_dict

In [ ]:
df_total = construct_total_df (json_val_path, seed=0, cluster_config="clusters4_layer12", cluster_numbers=[1,2, 3,4])
df_full = construct_df_full_data (json_val_path, seed=0, rename_columns="baseline")
diff_dict0 = get_performance_differences (df_total, df_full)
diff_dict0

In [ ]:
#df0 = pd.DataFrame.from_dict(diff_dict0, orient='index', columns = ["Values0"])
df1= pd.DataFrame.from_dict(list(diff_dict1.items()), columns = ["Metric", "Value"] )
# df_res =  pd.concat([df0, df1], axis=0)
# df_res
df1

In [ ]:
df1 = pd.DataFrame(list(diff_dict1.items()), columns=["Metric", "Value"])
df1

In [ ]:
diff_dict1.items()

In [ ]:
dict1_new = {}
for key in diff_dict1.keys():
    new_key = "_".join(key.split("_")[:-1])
    dict1_new [new_key] = diff_dict1[key]
dict1_new

In [ ]:
import os
import json
json_val_path = "json_val_scores"
df_total = construct_total_df (json_val_path, seed=1, cluster_config="clusters4_layer12", cluster_numbers=[1,2, 3,4])
df_full = construct_df_full_data (json_val_path, seed=1, rename_columns="baseline")
diff_dict1 = get_performance_differences (df_total, df_full)
diff_dict1

In [ ]:
def rename_keys_dict(d):
    dict_new = {}
    for key in d.keys():
        new_key = "_".join(key.split("_")[:-1])
        dict_new [new_key] = d[key]
    return dict_new
def get_total_results (seeds, json_val_path, cluster_config, cluster_numbers):
    df_list = []
    for seed in seeds:
        df_total = construct_total_df (json_val_path, seed=seed, cluster_config=cluster_config, cluster_numbers=cluster_numbers)
        df_full = construct_df_full_data (json_val_path, seed=seed, rename_columns="baseline")
        diff_dict = get_performance_differences (df_total, df_full)
        diff_dict_renamed = rename_keys_dict(diff_dict)
        #print(diff_dict_renamed)
        df = pd.DataFrame.from_dict(diff_dict_renamed, orient="index", columns=[f"Value_{seed}"])
        #df.index.name = "Metric"
        df_list.append(df)
    df_result =  pd.concat(df_list, axis=1)
    return df_result

In [ ]:
df_total = get_total_results (seeds = [0,1], json_val_path = "json_val_scores", cluster_config="clusters4_layer12", cluster_numbers=[1,2, 3,4])
df_total.mean(axis = 1)
df_total

In [ ]:
import os
import json
json_val_path = "json_val_scores"
df_total = construct_total_df (json_val_path, seed=0, cluster_config="clusters3_layer113", cluster_numbers=[1,2, 3])
df_full = construct_df_full_data (json_val_path, seed=0, rename_columns="baseline")
diff_dict = get_performance_differences (df_total, df_full)
diff_dict

In [ ]:
df_total.isna() 

In [ ]:
df2

In [ ]:
len(set(df1.index)  & set(df2.index))

In [ ]:
idncies_repeat = [ind for ind in df_total.index.to_list() if df_total.index.to_list().count(ind) >1]
len(idncies_repeat)

In [ ]:
mapping

In [ ]:
seed = 0
mapping = { col: f"{col}_full_data_seed{seed}" for col in df.columns}
df = df.rename(columns=mapping)
df

In [ ]:
import json
import os
import pandas as pd
json_val_path = "json_val_scores"
keys = {}

folder = os.path.join(json_val_path, "full_data")
file_scores = [file for file in os.listdir(folder) if "best-seed0" in file][0]

json_file = open(os.path.join(folder, file_scores))
json1_str = json_file.read()
json1_data = json.loads(json1_str)
clsuter_keys = set(json1_data.keys())
print(len(clsuter_keys))
df  = pd.DataFrame(json1_data).T
df

In [ ]:
json1_data

# HPO

In [ ]:
import optuna
def filter_optuna_study(study):
    df = study.trials_dataframe()
    df_filtered = df[df["state"].isin(["COMPLETE", "PRUNED"]) ]
    date_cols = [col for col in df_filtered.columns if "datetime" in col or "duration" in col]
    df_filtered = df_filtered.drop(columns = date_cols, axis = 1)

    df_filtered = df_filtered.reset_index()
    df_filtered = df_filtered.drop(columns = ["index"], axis = 1)
    df_filtered["value"] = df_filtered["value"]* 100
    print(f"Size of filtered df is {len(df_filtered)}")
    return df_filtered
from optuna.distributions import (
    UniformDistribution, LogUniformDistribution,
    IntUniformDistribution, CategoricalDistribution
)

def extract_search_space(study):
    space = {}
    for trial in study.trials:
        for name, dist in trial.distributions.items():
            if name in space:
                continue
            if hasattr(dist, "low"):
                space[name] = (dist.low, dist.high)
            elif isinstance(dist, CategoricalDistribution):
                space[name] = list(dist.choices)
    return space
def get_pruned(df, tail = 10):
    last_df = df.tail(tail)
    df_pruned_last = last_df[last_df["state"] == "PRUNED"]
    print(f"Percentage of pruned runs is {len(df_pruned_last) / len(last_df)* 100}% ")
def get_study_info(study_name, storage_path, baseline_value = None, pruned = True, tail= None):
    study = optuna.load_study(
        study_name=study_name,
        storage=storage_path
    )
    df_filtered = filter_optuna_study(study)
    search_space = extract_search_space(study)
    print(search_space)
    if pruned:
        get_pruned(df_filtered, tail = tail)
    if baseline_value is not None:
        
        df_filtered["diff"] = df_filtered["value"].apply(lambda x: x - baseline_value)
        
    indices = df_filtered["value"].nlargest(10).index
    df_filtered = df_filtered.loc[indices]
    return df_filtered, search_space
def print_top_trials_params(study_name, storage_path, top_k = 3):
    study = optuna.load_study(
        study_name=study_name,
        storage=storage_path
    )
    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:top_k]

    for i, t in enumerate(top_trials, 1):
        print(f"Rank {i}: Trial #{t.number}")
        print(f"  Value: {t.value}")
        print(f"  Params:")
        for k, v in t.params.items():
            print(f"    {k}: {v}")
        print("-----")
def print_top_k(study_name, storage_path, top_k):
    study = optuna.load_study(study_name=study_name, storage=storage_path)
    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:top_k]

    for i, t in enumerate(top_trials, 1):
        print(f"Rank {i}: Trial #{t.number}")
        print(f"  Value: {t.value}")
        print(f"  Params:")
        for k, v in t.params.items():
            print(f"    {k}: {v}")
        print("-----")

## Full data

In [ ]:
import optuna
import pandas as pd
study = optuna.load_study(
    study_name="hpo_full",
    storage="sqlite:///optuna.db"
)

# Convert to pandas DataFrame
df = study.trials_dataframe()
df_filtered = df[df["state"].isin(["COMPLETE", "PRUNED"]) ]
date_cols = [col for col in df_filtered.columns if "datetime" in col or "duration" in col]
df_filtered = df_filtered.drop(columns = date_cols, axis = 1)

df_filtered = df_filtered.reset_index()
df_filtered = df_filtered.drop(columns = ["index"], axis = 1)
df_filtered.head(10)

In [ ]:
import optuna
import pandas as pd

df_filtered, search_space = get_study_info(study_name="hpo_full", storage_path="sqlite:///optuna.db",
                                           baseline_value=64.813, tail = 10)
df_filtered

In [ ]:
indices = df_filtered["value"].nlargest(10).index
df_filtered.loc[indices]

In [ ]:
top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:3]

for i, t in enumerate(top_trials, 1):
    print(f"Rank {i}: Trial #{t.number}")
    print(f"  Value: {t.value}")
    print(f"  Params:")
    for k, v in t.params.items():
        print(f"    {k}: {v}")
    print("-----")

In [ ]:
ind = df_filtered["value"].argmax()
df_filtered.iloc[ind]

In [ ]:
last_df = df_filtered.tail(10)
last_df

In [ ]:
df_complete = df_filtered[df_filtered["state"] == "COMPLETE"]
last_complete_df = df_complete.tail(10)
last_complete_df

In [ ]:
def describe_column (df,column, multiply = False, complete = False):
    df_filtered = df[df["state"] == "COMPLETE"] if complete else df.copy()
    if multiply:
        values = df_filtered [column]*100
    else:
        values = df_filtered [column].copy()
    mean = values.mean()
    std =values.std()
    round_value = 2 if multiply else 5
    return round(mean, round_value ), round(std, round_value )

In [ ]:
columns = ["value", "params_batch_size", "params_lr", "params_weight_decay"]
for column in columns:
    multiply = True if column == "value" else False
    mean, std = describe_column(last_complete_df, column, multiply, complete=False)
    print(f"Results for column {column}")
    print(f"{mean} +- {std}")


In [ ]:
last_df_complete = last_df[last_df["state"] == "COMPLETE"]
res =last_df_complete["value"].describe()*100

In [ ]:
res["mean"]

In [ ]:
df_pruned_last = last_df[last_df["state"] == "PRUNED"]
print(f"Percentage of pruned runs is {len(df_pruned_last) / len(last_df)* 100}% ")

In [ ]:
from optuna.importance import get_param_importances, FanovaImportanceEvaluator
importances = []

for _ in range(10):
    imp = optuna.importance.get_param_importances(
        study, evaluator=FanovaImportanceEvaluator()
    )
    s = sum(imp.values())
    importances.append({k: v / s for k, v in imp.items()})

importance_avg = {
    k: sum(d.get(k, 0.0) for d in importances) / len(importances)
    for k in importances[0]
}
print(importance_avg)
print(sum(list(importance_avg.values())))

In [ ]:
from optuna.importance import get_param_importances, FanovaImportanceEvaluator
importance = optuna.importance.get_param_importances(
            study, evaluator=FanovaImportanceEvaluator()
        )
for i in range (9):
    new_importance = optuna.importance.get_param_importances(
            study, evaluator=FanovaImportanceEvaluator()
        )
    importance = {key: value + new_importance[key]  for key, value in importance.items()}
importance_norm = {key: value / 10 for key, value in importance.items()}
# total = sum(importance_norm.values())
# importance_norm = {k: v / total for k, v in importance_norm.items()}
print(importance_norm)

In [ ]:
sum(list(importance_norm.values()))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Assume df is already loaded with columns "dropout_top" and "eval/accuracy"
x = df_filtered["params_lr"].values
y = df_filtered["value"].values

# Fit linear regression
X = x.reshape(-1, 1)
model = LinearRegression().fit(X, y)
coef = model.coef_[0]
intercept = model.intercept_
r2 = model.score(X, y)

# Prepare data for plotting
order = np.argsort(x)
x_sorted = x[order]
y_pred_sorted = model.predict(x_sorted.reshape(-1, 1))

# Compute rolling mean for trend line
window = max(8, len(x) // 20)
y_smooth = pd.Series(y[order]).rolling(window= 20, min_periods=4, center=True).mean()

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(x, y, alpha=0.6, edgecolors='k', s=30)
#ax.plot(x_sorted, y_smooth, lw=2, label='Rolling Mean Trend')
ax.plot(x_sorted, y_pred_sorted, lw=2, label='Linear Regression Fit', color = "g")

ax.set_xlabel("Learing rate")
ax.set_xlim(0, 0.001)
ax.set_ylabel("Validation accuracy")
#ax.set_title("Dropout vs Eval/Accuracy with Linear Regression")
ax.legend()
#plt.savefig("figures/nas_frozen_layesrs.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

columns = ["val_F-measure_beat", "val_F-measure_downbeat", "Name"]
df = pd.read_csv("wandb_csvs/wandb_hpo_full.csv")
df = df.dropna(how='all')
df = df[df["State"] == "finished"]
df[columns].head()

In [ ]:
df.head()

In [ ]:
from scipy import stats
res_spearman = stats.spearmanr(df["val_F-measure_beat"].values, df["val_F-measure_downbeat"].values)
res_pearson = stats.pearsonr(df["val_F-measure_beat"].values, df["val_F-measure_downbeat"].values)
res_spearman, res_pearson

In [ ]:
df["val_F-measure_beat"].values

### validation accuracies

In [ ]:
import os
import json
import pandas as pd
def get_scores (json_path,  seed = 42, trial= None, return_means = False, folder_name ="hpo_full" ):
    folder = folder_name + f"_trial_{trial}" if trial else f"hpo_full_baseline"
   
    #folder = f"hpo_full_trial_{trial}" if trial else f"hpo_full_baseline"
    folder = os.path.join(json_path, folder)
    file_scores = [file for file in os.listdir(folder) if file.startswith(f"best-seed{seed}") ][0]

    json_file = open(os.path.join(folder, file_scores))
    json1_str = json_file.read()
    json1_data = json.loads(json1_str)

    #print(len(clsuter_keys))
    df  = pd.DataFrame(json1_data).T
    if return_means:
        return df.mean().to_dict(), df
    return df

In [ ]:
json_path = "json_val_scores"
trials = [22, 44, 12]
results_list = []
for trial in trials: 
    means, df = get_scores  (json_path,  seed = 42, trial= trial, return_means = True)
    assert len(df) == 556
    results_list.append(pd.DataFrame.from_dict(means, orient= "index", columns = [f"trial_{trial}"]).T)
means, df = get_scores  (json_path,  seed = 42, trial= None, return_means = True)
results_list.append(pd.DataFrame.from_dict(means, orient= "index", columns = [f"baseline"]).T)
result = pd.concat(results_list)
result
    

### joint hpo for beat and downbeat

In [ ]:
import optuna
import pandas as pd

df_filtered, search_space = get_study_info(study_name="full_data", storage_path="sqlite:///optuna_joint.db",
                                           tail = 10)
df_filtered

### joint evaluation

In [ ]:
json_path = "json_val_scores"
trials = [24, 31, 48]
results_list = []
for trial in trials: 
    means, df = get_scores  (json_path,  seed = 42, trial= trial, return_means = True, folder_name = "full_data_joint")
    assert len(df) == 556
    results_list.append(pd.DataFrame.from_dict(means, orient= "index", columns = [f"trial_{trial}"]).T)
means, df = get_scores  (json_path,  seed = 42, trial= None, return_means = True)
results_list.append(pd.DataFrame.from_dict(means, orient= "index", columns = [f"baseline"]).T)
result = pd.concat(results_list)
result

### deciding on checkpoint

for model from trial 12 (actually 9), the best result was achieved at epoch 74

In [ ]:
import pandas as pd 
import numpy as np
df = pd.read_csv("wandb_csvs/wandb_best_hpo_full.csv", index_col=False)
columns_acc = [col for col in df.columns.values if ("val_F" in col) & ("MAX" not in col) & ("MIN" not in col)]
df = df[  ["epoch"]  + columns_acc  ]
df.head(60)

In [ ]:
epochs = np.array([ 1, 5, 10, 15, 20, 30, 35, 40, 50, 60, 70, 75, 80, 90, 100] )-1
df_filtered = df[df["epoch"].isin( epochs)]
df_filtered["epoch"] = df_filtered["epoch"].apply(lambda x: x +1)
df_filtered

In [ ]:
import optuna
import pandas as pd
study = optuna.load_study(
    study_name="cluster_try",
    storage="sqlite:///optuna_try.db"
)

# Convert to pandas DataFrame
df = study.trials_dataframe()
df_filtered = df[df["state"].isin(["COMPLETE", "PRUNED"]) ]
date_cols = [col for col in df_filtered.columns if "datetime" in col or "duration" in col]
df_filtered = df_filtered.drop(columns = date_cols, axis = 1)

df_filtered = df_filtered.reset_index()
df_filtered = df_filtered.drop(columns = ["index"], axis = 1)
df_filtered.head(10)

## cluster specific hpo

In [ ]:
def filter_optuna_study(study):
    df = study.trials_dataframe()
    df_filtered = df[df["state"].isin(["COMPLETE", "PRUNED"]) ]
    date_cols = [col for col in df_filtered.columns if "datetime" in col or "duration" in col]
    df_filtered = df_filtered.drop(columns = date_cols, axis = 1)

    df_filtered = df_filtered.reset_index()
    df_filtered = df_filtered.drop(columns = ["index"], axis = 1)
    df_filtered["value"] = df_filtered["value"]* 100
    print(f"Size of filtered df is {len(df_filtered)}")
    return df_filtered
from optuna.distributions import (
    UniformDistribution, LogUniformDistribution,
    IntUniformDistribution, CategoricalDistribution
)

def extract_search_space(study):
    space = {}
    for trial in study.trials:
        for name, dist in trial.distributions.items():
            if name in space:
                continue
            if hasattr(dist, "low"):
                space[name] = (dist.low, dist.high)
            elif isinstance(dist, CategoricalDistribution):
                space[name] = list(dist.choices)
    return space
def get_pruned(df, tail = 10):
    last_df = df.tail(tail)
    df_pruned_last = last_df[last_df["state"] == "PRUNED"]
    print(f"Percentage of pruned runs is {len(df_pruned_last) / len(last_df)* 100}% ")
def get_study_info(study_name, storage_path, baseline_value = None, pruned = True, tail= None):
    study = optuna.load_study(
        study_name=study_name,
        storage=storage_path
    )
    df_filtered = filter_optuna_study(study)
    search_space = extract_search_space(study)
    print(search_space)
    if pruned:
        get_pruned(df_filtered, tail = tail)
    if baseline_value is not None:
        indices = df_filtered["value"].nlargest(10).index
        df_filtered["diff"] = df_filtered["value"].apply(lambda x: x - baseline_value)
        df_filtered = df_filtered.loc[indices]
    
    return df_filtered, search_space
def print_top_trials_params(study_name, storage_path, top_k = 3):
    study = optuna.load_study(
        study_name=study_name,
        storage=storage_path
    )
    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:top_k]

    for i, t in enumerate(top_trials, 1):
        print(f"Rank {i}: Trial #{t.number}")
        print(f"  Value: {t.value}")
        print(f"  Params:")
        for k, v in t.params.items():
            print(f"    {k}: {v}")
        print("-----")
def print_top_k(study_name, storage_path, top_k):
    study = optuna.load_study(study_name=study_name, storage=storage_path)
    top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:top_k]

    for i, t in enumerate(top_trials, 1):
        print(f"Rank {i}: Trial #{t.number}")
        print(f"  Value: {t.value}")
        print(f"  Params:")
        for k, v in t.params.items():
            print(f"    {k}: {v}")
        print("-----")

### 4 clusters

#### cluster 4

In [ ]:
import optuna
import pandas as pd

df_filtered, search_space = get_study_info(study_name="4clusters_cluster_4", storage_path="sqlite:///optuna_down.db",
                                           baseline_value=64.813, tail = 10)
df_filtered

In [ ]:
print_top_k(study_name="4clusters_cluster_4", storage_path="sqlite:///optuna_down.db", top_k=3)

In [ ]:
import optuna
import pandas as pd
study_name, storage_path = "4clusters_cluster_2", "sqlite:///optuna_down.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=93.186, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=3)
df_filtered

In [ ]:
import optuna
import pandas as pd
import optuna
import pandas as pd
study_name, storage_path = "4clusters_cluster_3", "sqlite:///optuna_down.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=95.236, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=3)
df_filtered

In [ ]:
study_name, storage_path = "4clusters_cluster_1", "sqlite:///optuna_down.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=92.51, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=3)
df_filtered

### 2 clusters

In [ ]:
# study = optuna.load_study(
#         study_name="2clusters_cluster2",
#         storage="sqlite:///optuna_new.db"
#     )
# optuna.delete_study(study_name="2clusters_cluster2", storage="sqlite:///optuna_new.db")

#### cluster 2

In [ ]:
import optuna
import pandas as pd
study_name, storage_path ="2clusters_cluster2_best", "sqlite:///optuna_best.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=88.878, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=1)
df_filtered

In [ ]:
import optuna
study = optuna.load_study(
        study_name="2clusters_cluster2",
        storage="sqlite:///optuna_new.db"
    )
trial = study.trials[69]
print(trial.params)

In [ ]:
import optuna
import pandas as pd
study_name, storage_path ="2clusters_cluster2", "sqlite:///optuna_new.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=88.878, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=1)
df_filtered


In [ ]:
import optuna
import pandas as pd
import optuna
import pandas as pd
study_name, storage_path = "cluster_2_30_ep", "sqlite:///optuna_new.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=88.878, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=1)
df_filtered


In [ ]:
import optuna
import pandas as pd


df_filtered, search_space = get_study_info(study_name="cluster_2_best_ckpt", storage_path="sqlite:///optuna_new.db",
                                           baseline_value=88.878, tail = 10)
df_filtered

In [ ]:
import optuna
import pandas as pd


df_filtered, search_space = get_study_info(study_name="cluster_2_ckpt_best", storage_path="sqlite:///optuna.db",
                                           baseline_value=88.878, tail = 10)
df_filtered

In [ ]:
import optuna
import pandas as pd
df_filtered, search_space = get_study_info(study_name="cluster_2_30_ep", storage_path="sqlite:///optuna_new.db",
                                           baseline_value=88.878, tail =10 )
df_filtered

In [ ]:
study = study = optuna.load_study(
        study_name="cluster_2_30_ep",
        storage="sqlite:///optuna_new.db"
    )
top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:3]
top_trial = top_trials[1]

for i, t in enumerate(top_trials, 1):
    print(f"Rank {i}: Trial #{t.number}")
    print(f"  Value: {t.value}")
    print(f"  Params:")
    print(t.params["lr"])
    # for k, v in t.params.items():
    #     print(f"    {k}: {v}")
    # print("-----")

In [ ]:
print_top_trials_params(study_name="cluster_2_30_ep", storage_path="sqlite:///optuna_new.db", top_k=3)

In [ ]:
top_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else -float("inf"), reverse=True)[:3]

for i, t in enumerate(top_trials, 1):
    print(f"Rank {i}: Trial #{t.number}")
    print(f"  Value: {t.value}")
    print(f"  Params:")
    for k, v in t.params.items():
        print(f"    {k}: {v}")
    print("-----")

In [ ]:
import optuna
import pandas as pd

def load_df(ckpt_epoch, cluster):
    study_name = f"cluster_{cluster}_ckpt{ckpt_epoch}"

    study = optuna.load_study(
        study_name=study_name,
        storage="sqlite:///optuna.db"
    )
    df= study.trials_dataframe()
    return df

In [ ]:
def filter_df (df, return_top = None, ckpt_epoch = None):
    df_filtered = df[df["state"].isin(["COMPLETE", "PRUNED"]) ]
    date_cols = [col for col in df_filtered.columns if "datetime" in col or "duration" in col]
    df_filtered = df_filtered.drop(columns = date_cols, axis = 1)

    df_filtered = df_filtered.reset_index()
    df_filtered = df_filtered.drop(columns = ["index"], axis = 1)
    if ckpt_epoch:
        df_filtered["ckpt epoch"] = [ckpt_epoch] * len(df_filtered)
    if return_top:
        indices = df_filtered["value"].nlargest(return_top).index
        df_top= df_filtered.loc[indices]#
        return  df_filtered, df_top
    return df_filtered

In [ ]:
def combine_dfs (df_list, sort = False):
    df_result =  pd.concat(df_list, axis=0)
    if sort:
        df_result =df_result.sort_values(by= ["value"], ascending = False)
    return df_result

In [ ]:
import optuna
import pandas as pd
study = optuna.load_study(
    study_name="cluster_2_ckpt10",
    storage="sqlite:///optuna.db"
)
df_10= study.trials_dataframe()
df_10,df_10_top = filter_df(df_10, return_top=3, ckpt_epoch=10)
#df_10_top["ckpt_epoch"] = [10]* len(df_10_top)
df_10_top

In [ ]:
ckpt_epochs = [10, 40, 60, "_best"]
df_len = {}
df_list = []
for ckpt in ckpt_epochs:
    df = load_df(ckpt_epoch= ckpt, cluster =2)
    filtered_df, df_top = filter_df(df, return_top=3, ckpt_epoch=ckpt)
    df_list.append(df_top)
    df_len[ckpt] = len(filtered_df)
df_result = combine_dfs(df_list, sort= True ) 
df_result

In [ ]:
df_len

#### cluster 1

In [ ]:

import optuna
import pandas as pd
import optuna
import pandas as pd
study_name, storage_path ="2clusters_cluster1", "sqlite:///optuna_new.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=97.045, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=1)
df_filtered


In [ ]:
import optuna
import pandas as pd
import optuna
import pandas as pd
study_name, storage_path = "cluster_1__lr", "sqlite:///optuna_new.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=97.045, tail = 10)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=1)
df_filtered


In [ ]:
import optuna
import pandas as pd
study_name = "cluster_1__lr"
storage_path = "sqlite:///optuna_new.db"
df_filtered, search_space = get_study_info(study_name=study_name, storage_path=storage_path,
                                           baseline_value=97.045)
print_top_k(study_name=study_name, storage_path=storage_path, top_k=3)
df_filtered

In [ ]:
print_top_k(study_name="cluster_1_best", storage_path="sqlite:///optuna_new.db", top_k=3)

In [ ]:
import pandas as pd

columns = ["val_F-measure_beat", "val_F-measure_downbeat", "Name"]
df = pd.read_csv("wandb_csvs/wandb_cluster1.csv")
df = df.dropna(how='all')
df = df[df["State"] == "finished"]
df = df[columns]
df.head()

In [ ]:
downbeat_baseline = 0.9341731667518616
df["diff_downbeat"] = df["val_F-measure_downbeat"].apply(lambda x: x - downbeat_baseline)
df_sorted = df.sort_values(by=["diff_downbeat"], ascending=False)
df_sorted

## clsuter specific evaluation

#### clusters 4

##### cluster 1

In [ ]:
df_total = get_per_cluster_results(trials = [10, 15, 24], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [10], seeds = [0,1,2,3,4,5,6,7,8,9],json_path = "json_test_scores", name_full = "full_model_trial12", cluster = 1,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
df_total
#0,1,2,3,4,5,6,7,8,9

In [ ]:
trial= 10
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
cluster = 1
cluster_config="clusters4_layer12"
scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
                    df_path=df_path, full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}"
                
                )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals =4)
    
else:
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
res

In [ ]:
json_path = "json_test_scores" 
trial= 10
name = ""
decimals = 3
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster = 1
cluster_config="clusters4_layer12"
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
scores, genres = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
            df_path=df_path, full=False, per_cluster=True, name = f"cluster_{cluster}_trial_{trial}{name}"
        
        )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals)
res

In [ ]:
json_path = "json_val_scores" 
name = ""
decimals = 3
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster = 2
cluster_config="clusters4_layer12"
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
scores, genres = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds,  subfolder = "all_data_metrics", filter_files=True,
            df_path=df_path, full=False, name = f"paper_baseline_99"
        
        )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals, name = f"paper_baseline_99")
res
columns =["F-measure", "AMLt","CMLt", "ACR_L2_any"]
cols = [col for col in res.columns if any(c in col for c in columns)]
res = res[cols[:-1]]
res

In [ ]:
json_path = "json_test_scores" 
trial= 10
name = ""
decimals = 3
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster = 1
cluster_config="clusters4_layer12"
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
scores, genres = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config, subfolder = "all_data_metrics", filter_files=True,
            df_path=df_path, full=False, name = f"cluster_{cluster}_trial_{trial}"
        
        )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals)
res
columns =["F-measure", "AMLt","CMLt", "ACR_L2_any"]
cols = [col for col in res.columns if any(c in col for c in columns)]
res = res[cols[:-1]]
res

In [ ]:
json_path = "json_test_scores" 
trial= 10
name = ""
decimals = 3
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster = 1
cluster_config="clusters4_layer12"
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
scores, genres = get_restults_per_cluster(
            json_path, cluster=cluster, seeds=seeds, subfolder = "all_data_metrics", filter_files=True,
            df_path=df_path, full=False, name = "paper_baseline_99"
            
        
        )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals)
res
columns =["F-measure", "AMLt","CMLt", "ACR_L2_any"]
cols = [col for col in res.columns if any(c in col for c in columns)]
res = res[cols[:-1]]
res

In [ ]:
json_path = "json_test_scores" 
trial= 10
name = ""
decimals = 3
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster = 1
cluster_config="clusters4_layer12"
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
scores, genres = get_restults_per_cluster(
            json_path, cluster=cluster, all_metrics=True, seeds=seeds, cluster_config=cluster_config,
            df_path=df_path, full=False, per_cluster=True, name = f"cluster_{cluster}_trial_{trial}"
        
        )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals)
res

##### cluster 2

In [ ]:
df_total = get_per_cluster_results(trials = [19, 5, 16], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 2,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [16], seeds = [5],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 2,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

##### cluster 3

In [ ]:
df_total = get_per_cluster_results(trials = [75, 82, 72], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 3,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [72], seeds = [0,1,2,3,4,5,6,7,8,9],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 3,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

In [ ]:
trial= 72
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
cluster = 3
scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config="clusters4_layer12",
                    df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}"
                
                )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals =4)
    
else:
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
res

In [ ]:
trial= 72
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
cluster = 3
scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config="clusters4_layer12",
                    df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv", full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}", subfolder = "all_data_metrics"
                
                )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals =4)
    
else:
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
res

##### cluster 4

In [ ]:
df_total = get_per_cluster_results(trials = [50, 43, 14], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 4,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [43], seeds = [0,1,2,3,4,5,6,7,8,9],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 4,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [43], seeds = [0,1, 2,3,4],json_path = "json_test_scores", name_full = "full_model_trial12", cluster = 4,
                                    cluster_config="clusters4_layer12", df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

### 2 clusters

#### cluster 1

In [ ]:
#new
df_total = get_per_cluster_results(trials = [47,66, 65], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1)
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [47, ], seeds = [0,1,2,3,4,5,6,7,8,9],  df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv",
                                   json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1, cluster_config = "clusters2_layer6",
                                   extrat_std=True, decimals=2)
rows = df_to_latex_rows(df_total, rename_idx=["Cluster 1", "Baseline (1)"])
for r in rows:
    print(r)
df_total

In [ ]:
trial= 47
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
cluster = 1
cluster_config = "clusters2_layer6"
df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv"
scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
                    df_path=df_path, full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}", subfolder = "all_data_metrics"
                
                )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals =4)
    
else:
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
res

In [ ]:
df_total = get_per_cluster_results(trials = [47, ], seeds = [0,1,2,3,4,5,6,7,8,9],  df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv",
                                   json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1, cluster_config = "clusters2_layer6",
                                   extrat_std=True, decimals=2)
rows = df_to_latex_rows(df_total, rename_idx=["Cluster 1", "Baseline (1)"], bold_difference=True, bold_baseline_if_best= True)
for r in rows:
    print(r)
df_total

In [ ]:
df_total = get_per_cluster_results(trials = [47, ], seeds = [0,1,2,3,4,5,6,7,8,9],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1)
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

In [ ]:
get_latex_row(diff_vs_baseline)

In [ ]:
# old
df_total = get_per_cluster_results(trials = [39, 42, 44], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1)
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
df_total = get_per_cluster_results(trials = [39, ], seeds = [0,1, 2],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 1)
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline
diff_vs_baseline

#### cluster 2

In [ ]:
# new study
df_total = get_per_cluster_results(trials = [61, 85, 69], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 2,
                                    cluster_config="clusters2_layer6", df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline
diff_vs_baseline

In [ ]:
# new study
df_total = get_per_cluster_results(trials = [69], seeds = [0,1,2,3,4,5,6,7,8,9],json_path = "json_test_scores", name_full = "full_model_trial12", cluster = 2,
                                    cluster_config="clusters2_layer6", df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv")
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=1)
diff_vs_baseline

In [ ]:
trial= 69
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
cluster = 2
cluster_config = "clusters2_layer6"
df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv"
scores, genres = get_restults_per_cluster(
                    json_path, cluster=cluster, seeds=seeds, cluster_config=cluster_config,
                    df_path=df_path, full=False, filter_files=True, name = f"cluster_{cluster}_trial_{trial}", subfolder = "all_data_metrics"
                
                )
if len(seeds) >1:
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals =4)
    
else:
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
res

In [ ]:
df_total = get_per_cluster_results(trials = [69, ], seeds = [0,1,2,3,4,5,6,7,8,9],  df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv",
                                   json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 2, 
                                   extrat_std=True, decimals=2)
rows = df_to_latex_rows(df_total, rename_idx=["Cluster 2", "Baseline (2)"])
for r in rows:
    print(r)
df_total

In [ ]:
#old study (30 ep)
df_total = get_per_cluster_results(trials = [34, 17, 15], seeds = [0],json_path = "json_val_scores", name_full = "full_model_trial12", cluster = 2)
diff_vs_baseline = get_baseliine_diff (df_total, num_trials=3)
diff_vs_baseline

In [ ]:
json_val_path = "json_val_scores"
import os
import pandas as pd
import json
trials = [34, 17, 15]
seeds = [42]
df_list = []
for trial in trials:
    scores, genres = get_restults_per_cluster(
                json_val_path, cluster=2, seeds=[42], cluster_config="clusters2_layer6",
                df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = f"cluster_2_trial_{trial}"
            )
    res = pd.DataFrame.from_dict(scores[0]  , orient="index", columns=[f"trial_{trial}"])
    res = pd.DataFrame(res).T 
    df_list.append(res)

scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[42], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "hpo_full_trial_12"
        )
res_base = pd.DataFrame.from_dict(scores_base[0]  , orient="index", columns=[f"baseline"])
res_base = pd.DataFrame(res_base).T
df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
diff_vs_baseline = df_total.subtract(df_total.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

#### all 3 different trials with seed 0

In [ ]:
json_val_path = "json_val_scores"
import os
import pandas as pd
import json
trials = [34, 15, 17]
seeds = [ 0]
df_list = []
for trial in trials:
    scores, genres = get_restults_per_cluster(
                json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
                df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = f"cluster_2_trial_{trial}"
            )
    mean_dict = mean_scores(scores)

    res = pd.DataFrame.from_dict(
        mean_dict,
        orient="index",
        columns=[f"trial_{trial}"]
    ).T
    # res = pd.DataFrame.from_dict(scores_cl2[0]  , orient="index", columns=[f"trial_{trial}"])
    # res = pd.DataFrame(res).T 
    df_list.append(res)

scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "full_model_trial12"
        )
mean_base = pd.DataFrame(scores_base).mean().to_dict()

res_base = pd.DataFrame.from_dict(
    mean_base,
    orient="index",
    columns=["baseline"]
).T

df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
diff_vs_baseline = df_total.subtract(df_total.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

In [ ]:
json_val_path = "json_val_scores"
import os
import pandas as pd
import json
trials = [34]
seeds = [ 0,1,2,3,4 ]
df_list = []
for trial in trials:
    scores, genres = get_restults_per_cluster(
                json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
                df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = f"cluster_2_trial_{trial}"
            )
    mean_dict = mean_scores(scores)

    res = pd.DataFrame.from_dict(
        mean_dict,
        orient="index",
        columns=[f"trial_{trial}"]
    ).T
    # res = pd.DataFrame.from_dict(scores_cl2[0]  , orient="index", columns=[f"trial_{trial}"])
    # res = pd.DataFrame(res).T 
    df_list.append(res)

scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "full_model_trial12"
        )
mean_base = pd.DataFrame(scores_base).mean().to_dict()

res_base = pd.DataFrame.from_dict(
    mean_base,
    orient="index",
    columns=["baseline"]
).T

df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
diff_vs_baseline = df_total.subtract(df_total.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

#### test scores


In [ ]:
json_val_path = "json_test_scores" # only for the baseline itself on the full data
import os
import pandas as pd
import json
trials = [34]
seeds = [ 0,1, 2, 3, 4]
df_list = []

scores_base = get_restults_per_cluster(
            json_val_path, seeds=seeds, 
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=False, cluster = 2, name = "full_model_trial12"
        )
mean_base = pd.DataFrame(scores_base).mean().to_dict()

res_base = pd.DataFrame.from_dict(
    mean_base,
    orient="index",
    columns=["baseline"]
).T

df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
json_val_path = "json_test_scores"
import os
import pandas as pd
import json
trials = [34]
seeds = [ 0,1, 2, 3, 4]
df_list = []
for trial in trials:
    scores, genres = get_restults_per_cluster(
                json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
                df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = f"cluster_2_trial_{trial}"
            )
    mean_dict = mean_scores(scores)

    res = pd.DataFrame.from_dict(
        mean_dict,
        orient="index",
        columns=[f"trial_{trial}"]
    ).T
    # res = pd.DataFrame.from_dict(scores_cl2[0]  , orient="index", columns=[f"trial_{trial}"])
    # res = pd.DataFrame(res).T 
    df_list.append(res)

scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "full_model_trial12"
        )
mean_base = pd.DataFrame(scores_base).mean().to_dict()

res_base = pd.DataFrame.from_dict(
    mean_base,
    orient="index",
    columns=["baseline"]
).T

df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
diff_vs_baseline = df_total.subtract(df_total.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

In [ ]:
json_file = open("json_val_scores/clusters2_layer6/2/cluster_2_trial_15/best-seed2-epoch=03-valfval_F-measure_beat=0.8908_orig.json")
json1_str = json_file.read()
json1_data = json.loads(json1_str)
cluster_keys = set(json1_data.keys())
keys_b = [key for key in cluster_keys if "beatles" in key]
keys_b
#json1_data["beats/beatles_10_CD2_The_Beatles_12_Revolution_9"]

In [ ]:
json_val_path = "json_test_scores"
import os
import pandas as pd
import json
trials = [ 34]
seeds = [0,1, 2]
df_list = []
for trial in trials:
    scores, genres = get_restults_per_cluster(
                json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
                df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = f"cluster_2_trial_{trial}"
            )
    mean_dict = mean_scores(scores)

    res = pd.DataFrame.from_dict(
        mean_dict,
        orient="index",
        columns=[f"trial_{trial}"]
    ).T
    # res = pd.DataFrame.from_dict(scores_cl2[0]  , orient="index", columns=[f"trial_{trial}"])
    # res = pd.DataFrame(res).T 
    df_list.append(res)

scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=seeds, cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "full_model_trial12"
        )
mean_base = pd.DataFrame(scores_base).mean().to_dict()

res_base = pd.DataFrame.from_dict(
    mean_base,
    orient="index",
    columns=["baseline"]
).T

df_list.append(res_base)
df_total = pd.concat(df_list, axis=0)
df_total

In [ ]:
diff_vs_baseline = df_total.subtract(df_total.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

In [ ]:
scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[42], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = "full_model_trial12"
        )

In [ ]:
scores_base = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[0], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = "full_model_trial12"
        )

In [ ]:
json_val_path = "json_val_scores"
scores = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[42], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "hpo_full_trial_12"
        )
scores[0]

In [ ]:
json_val_path = "json_val_scores"
scores = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[0], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=True, per_cluster=True, name = "full_model_trial12"
        )
scores

In [ ]:
json_val_path = "json_val_scores"
scores = get_restults_per_cluster(
            json_val_path, cluster=2, seeds=[42], cluster_config="clusters2_layer6",
            df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", full=False, per_cluster=True, name = "full_model_trial12"
        )
scores

## all data

In [ ]:
def extract_mean(series):
    return series.str.split("±").str[0].astype(float)

In [ ]:
seeds = [0,1,2]
names =["full_data_99"]
subfolders = ["full_data"]
json_path = "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names, best= False, subfolders=subfolders ) #rename_columns="baseline"
configs = {
    
    "baseline 99": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)

final_df_99 = remove_cemgil(final_df_99)
for row in df_to_latex_rows(final_df_99, rename_idx=[ "Baseline 99"], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col= False):
    print(row) 
final_df_99

In [ ]:
seeds = [0,1,2]
subfolders = ["full_data"]
json_path = "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True,  best= True, subfolders=subfolders ) #rename_columns="baseline"
configs = {
    
    "baseline early": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)

final_df_99 = remove_cemgil(final_df_99)
for row in df_to_latex_rows(final_df_99, rename_idx=[ "Baseline hpo early"], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col= False):
    print(row) 
final_df_99

In [ ]:
seeds = [0,1,2]
names =["full_model_trial12"]
subfolders = ["full_data"]
json_path = "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names, best= True, subfolders=subfolders ) #rename_columns="baseline"
configs = {
    
    "baseline hpo best": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)

final_df_99 = remove_cemgil(final_df_99)
for row in df_to_latex_rows(final_df_99, rename_idx=[ "Baseline hpo best"], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col= False):
    print(row) 
final_df_99

In [ ]:
seeds = [0,1,2]
json_val_path =  "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , full = True ) #rename_columns="baseline"
# df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 6, clusters = 2)
# df_seeds_mean, df_seeds_std, df_final_clusters4_layer12 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 12, clusters = 4)
# df_seeds_mean, df_seeds_std, df_final_clusters3_layer113= get_total_results_new(seeds =seeds , json_val_path = json_val_path , layer = 113, clusters = 3)
configs = {
    # "clusters2_layer6": df_final_clusters2_layer6,
    # "clusters4_layer12": df_final_clusters4_layer12,
    
    # "clusters3_raw" : df_final_clusters3_layer113,
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [50]:
seeds = [0,1,2,3,4,5,6,7,8,9]
json_val_path =  "json_val_scores"
subfolders = ["full_data"]
names = ["full_model_trial12"]

df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , full = True, names = names, best= True, subfolders=subfolders ) #rename_columns="baseline"
configs = {
    "baseline_hpo_trial_12": df_final_baseline
}
final_df_hpo = pd.DataFrame(configs).T 
final_df_hpo = remove_cemgil(final_df_hpo)
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
for row in df_to_latex_rows(final_df_hpo, rename_idx=[  "Baseline (HPO)" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_hpo

json_val_scores/full_data/full_model_trial12
best-seed0
json_val_scores/full_data/full_model_trial12/best-seed0-epoch=85-valfval_F-measure_beat=0.9379_orig.json
556
\textbf{Baseline (HPO)} & $92.85 \pm 0.10$ & $85.28 \pm 0.24$ & $89.09 \pm 0.27$ & $85.45 \pm 0.21$ & $72.94 \pm 0.55$ & $80.18 \pm 0.61$ \\


,F-measure_beat,CMLt_beat,AMLt_beat,F-measure_downbeat,CMLt_downbeat,AMLt_downbeat
baseline_hpo_trial_12,92.85 ± 0.10,85.28 ± 0.24,89.09 ± 0.27,85.45 ± 0.21,72.94 ± 0.55,80.18 ± 0.61


In [ ]:
seeds = [0,1, 2]
json_val_path =  "json_val_scores"
subfolders = ["full_data"]
names = ["full_model_trial12"]

df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , full = True, names = names, best= True, subfolders=subfolders ) #rename_columns="baseline"
configs = {
    # "clusters2_layer6": df_final_clusters2_layer6,
    # "clusters4_layer12": df_final_clusters4_layer12,
    
    # "clusters3_raw" : df_final_clusters3_layer113,
    "baseline_hpo_trial_12": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
final_df = remove_cemgil(final_df)
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["full_data_99"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names, best= False ) #rename_columns="baseline"
configs = {
    
    "baseline 99": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_99
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["full_data_99"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names, best= False ) #rename_columns="baseline"
configs = {
    
    "baseline 99": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_99
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1

In [ ]:
json_path = "json_val_scores" 
name = ""
decimals = 2
seeds = [0,1,2,3,4,5,6,7,8,9]
extrat_std = True
cluster_config="clusters4_layer12"
cluster = 1
df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
for cluster in [1,2,3,4]:
    scores, genres = get_restults_per_cluster(
                json_path, cluster=cluster, seeds=seeds,  subfolder = "all_data_metrics", filter_files=True,
                df_path=df_path, full=False, name = f"paper_baseline_99"
            
            )
    if len(seeds) >1:
        res = extract_mean_std(scores, trial = trial, extract_std=extrat_std, decimals = decimals, name = f"paper_baseline_99")
    res
    columns =["F-measure", "AMLt","CMLt", ]
    cols = [col for col in res.columns if any(c in col for c in columns)]
    res = res[cols[:-1]]
    for row in df_to_latex_rows(res, rename_idx=[f"Full 99"], bold_index = True, pm= True):
            print ("& "+ row)
            #print(row) 

#### all data metrics

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["paper_baseline_99"]
json_path = "json_val_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names, best= False, subfolders=["all_data_metrics"] ) #rename_columns="baseline"
configs = {
    
    "baseline 99": df_final_baseline
}
final_df_99 = pd.DataFrame(configs).T 
columns =["F-measure", "AMLt","CMLt",]
cols = [col for col in final_df_99.columns if any(c in col for c in columns)]
final_df_99 = final_df_99[cols[:-1]]
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
for row in df_to_latex_rows(final_df_99, rename_idx=[  "Baseline" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_99
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
json_val_path =  "json_test_scores"
names = ["paper_baseline_best"]
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , names = names, full = True, best=False, subfolders=["all_data_metrics"] ) #rename_columns="baseline"
configs = {
    "baseline_best": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
final_df

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
json_val_path =  "json_val_scores"
names = ["baseline_hpo_trial_12_best"]
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , names = names, full = True, best=False, subfolders=["all_data_metrics"] ) #rename_columns="baseline"
configs = {
    "baseline (hpo)": df_final_baseline
}
final_df_hpo = pd.DataFrame(configs).T 
final_df_hpo

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
json_val_path =  "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , full = True, best=True ) #rename_columns="baseline"
# df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 6, clusters = 2)
# df_seeds_mean, df_seeds_std, df_final_clusters4_layer12 = get_total_results_new(seeds = seeds, json_val_path = json_val_path , layer = 12, clusters = 4)
# df_seeds_mean, df_seeds_std, df_final_clusters3_layer113= get_total_results_new(seeds =seeds , json_val_path = json_val_path , layer = 113, clusters = 3)
configs = {
    # "clusters2_layer6": df_final_clusters2_layer6,
    # "clusters4_layer12": df_final_clusters4_layer12,
    
    # "clusters3_raw" : df_final_clusters3_layer113,
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
#final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["full_model_trial12"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names ) #rename_columns="baseline"
configs = {
    
    "baseline (hpo)": df_final_baseline
}
final_df_hpo = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_hpo
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)

In [ ]:
eeds = [0,1,2,3,4,5,6,7,8,9]
json_val_path =  "json_test_scores"
names = ["baseline_hpo_trial_12_best"]
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_val_path , names = names, full = True, best=False, subfolders=["all_data_metrics"] ) #rename_columns="baseline"
configs = {
    "baseline (hpo)": df_final_baseline
}
final_df_hpo = pd.DataFrame(configs).T 
final_df_hpo

In [ ]:
df = pd.concat([final_df, final_df_99, final_df_hpo], axis = 0) # val set
df = remove_cemgil(df)
df

In [ ]:
df = pd.concat([final_df, final_df_hpo], axis = 0) # test set
df = remove_cemgil(df)
df

In [ ]:
df = pd.concat([final_df, final_df_99, final_df_hpo], axis = 0) # val set
df = remove_cemgil(df)
df

#### 2 clusters

In [ ]:
def extract_mean(series):
    return series.str.split("±").str[0].astype(float)

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
# 47 and 69 are new, 39 and 34 are old
names =["full_model_trial12", "cluster_1_trial_47", "cluster_2_trial_69"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
                                                                       full = True, names = names, decimals = 2) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters2_layer12 = get_total_results_new(seeds = seeds, json_val_path = json_path, skip_cluster=[],
                                                                               layer = 6, clusters = 2, names = names, df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", decimals = 2 )
configs = {
    "clusters6_layer12": df_final_clusters2_layer12,
    
   # "baseline": df_final_baseline
}
final_df_2 = pd.DataFrame(configs).T 
final_df_2 = remove_cemgil(final_df_2)
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_2
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["full_model_trial12", "cluster_1_trial_47", "cluster_2_trial_69"]
json_path = "json_val_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters2_layer12 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 6, clusters = 2, names = names,  
         skip_cluster = [], df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", decimals=2,
          subfolders=["full_data", ""])
configs = {
    "clusters2_layer6": df_final_clusters2_layer12,
    
    #"baseline": df_final_baseline
}
final_df_2 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_2 = remove_cemgil(final_df_2)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_2
for row in df_to_latex_rows(final_df_2, rename_idx=[  "2" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_2

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["baseline_hpo_trial_12_best", "cluster_1_trial_47", "cluster_2_trial_69"]
json_path = "json_test_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 6, clusters = 2, names = names,  df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv", decimals=2,
          subfolders=["all_data_metrics", "all_data_metrics"] )
configs = {
    "clusters2_layer6": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_2 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_2 = remove_cemgil(final_df_2)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_2

In [ ]:
df = pd.concat([final_df, final_df_99, final_df_hpo, final_df_2], axis = 0) # val set
df = remove_cemgil(df)
df

In [ ]:
df_to_latex_rows(final_df, rename_idx=["Total (clusters)", "Total (baseline)"], bold_index = True)
for row in df_to_latex_rows(final_df, rename_idx=["Total (clusters)", "Total (baseline)"], bold_index = True):
    print(row)

In [ ]:
get_latex_row(final_df)

In [ ]:
df_means_only = final_df.apply(extract_mean, axis=1)
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
print(names)
print(json_path)
diff_vs_baseline

In [ ]:
a = list(np.arange(1,4+1))
a.append(None)
a

In [ ]:
get_latex_row(final_df)

In [ ]:
seeds = [0,1, 2,3,4]
# 47 and 69 are new, 39 and 34 are old
names =["full_model_trial12", "cluster_1_trial_47", "cluster_2_trial_69"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, full = True, names = names ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters2_layer6 = get_total_results_new(seeds = seeds, json_val_path = json_path, layer = 6, clusters = 2, names = names )
configs = {
    "clusters2_layer6": df_final_clusters2_layer6,
    
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)

In [ ]:
df_means_only = final_df.apply(extract_mean, axis=1)
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
print(names)
print(json_path)
diff_vs_baseline

#### 4 clusters

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["full_model_trial12", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_test_scores"
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
                                                                       full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  skip_cluster = [3], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=4 )
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["full_model_trial12", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_test_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  
         skip_cluster = [], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=4,
          subfolders=["full_data", ""] )
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4

In [69]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["full_model_trial12", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_val_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  
         skip_cluster = [3], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=2,
          subfolders=["full_data", ""])
configs = {
    "clusters4_layer12 wt3": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4_wt3 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4_wt3 = remove_cemgil(final_df_4_wt3 )
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4_wt3 
for row in df_to_latex_rows(final_df_4_wt3 , rename_idx=[  "4 (wt 3)" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_4_wt3 

using file   json_val_scores/clusters4_layer12/1/cluster_1_trial_10/best-seed0-epoch=25-valfval_F-measure_downbeat=0.9199_orig.json
number of files (220, 8)
(220, 8)
using file   json_val_scores/clusters4_layer12/2/cluster_2_trial_16/best-seed0-epoch=19-valfval_F-measure_downbeat=0.9332_orig.json
number of files (123, 8)
(123, 8)
skipping cluster 3
using file   json_val_scores/full_data/full_model_trial12/best-seed0-epoch=85-valfval_F-measure_beat=0.9379_orig.json
number of files (153, 8)
using file   json_val_scores/clusters4_layer12/4/cluster_4_trial_43/best-seed0-epoch=24-valfval_F-measure_downbeat=0.6557_orig.json
number of files (106, 8)
(106, 8)
\textbf{4 (wt 3)} & $92.82 \pm 0.09$ & $85.18 \pm 0.19$ & $89.04 \pm 0.23$ & $85.69 \pm 0.26$ & $73.48 \pm 0.51$ & $80.66 \pm 0.33$ \\


,F-measure_beat,CMLt_beat,AMLt_beat,F-measure_downbeat,CMLt_downbeat,AMLt_downbeat
clusters4_layer12 wt3,92.82 ± 0.09,85.18 ± 0.19,89.04 ± 0.23,85.69 ± 0.26,73.48 ± 0.51,80.66 ± 0.33


In [47]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["full_model_trial12", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_val_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  
         skip_cluster = [], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=2,
          subfolders=["full_data", ""])
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4 )
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4
for row in df_to_latex_rows(final_df_4 , rename_idx=[  "4" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_4

using file   json_val_scores/clusters4_layer12/1/cluster_1_trial_10/best-seed0-epoch=25-valfval_F-measure_downbeat=0.9199_orig.json
number of files (220, 8)
(220, 8)
using file   json_val_scores/clusters4_layer12/2/cluster_2_trial_16/best-seed0-epoch=19-valfval_F-measure_downbeat=0.9332_orig.json
number of files (123, 8)
(123, 8)
using file   json_val_scores/clusters4_layer12/3/cluster_3_trial_72/best-seed0-epoch=01-valfval_F-measure_downbeat=0.9496_orig.json
number of files (153, 8)
(153, 8)
using file   json_val_scores/clusters4_layer12/4/cluster_4_trial_43/best-seed0-epoch=24-valfval_F-measure_downbeat=0.6557_orig.json
number of files (106, 8)
(106, 8)
\textbf{4} & $92.81 \pm 0.10$ & $85.13 \pm 0.19$ & $88.99 \pm 0.25$ & $85.69 \pm 0.26$ & $73.48 \pm 0.48$ & $80.66 \pm 0.28$ \\


,F-measure_beat,CMLt_beat,AMLt_beat,F-measure_downbeat,CMLt_downbeat,AMLt_downbeat
clusters4_layer12,92.81 ± 0.10,85.13 ± 0.19,88.99 ± 0.25,85.69 ± 0.26,73.48 ± 0.48,80.66 ± 0.28


In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["paper_baseline_99", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"] #"baseline_hpo_trial_12_best", "paper_baseline_99"
json_path = "json_test_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  
         skip_cluster = [], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=2,
          subfolders=["all_data_metrics", "all_data_metrics"] )
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
# columns =["F-measure", "AMLt","CMLt", "ACR_L3_any"]
# cols = [col for col in final_df_4.columns if any(c in col for c in columns)]
#final_df_4 = final_df_4[cols[:-1]]
# for row in df_to_latex_rows(final_df_4, rename_idx=["4"], bold_index = True):
#     print(row)
columns =["ACR_L2_any", "ACR_L3_any", "ACR_L4_any"]
cols = [col for col in final_df_4.columns if any(c in col for c in columns)]
final_df_4= final_df_4[cols]
for row in df_to_latex_rows(final_df_4, rename_idx=[  "4" ], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col=False):
    print(row)
final_df_4


In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["paper_baseline_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_test_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  skip_cluster = [], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=4, all_metrics=True)
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4[["F-measure_beat", "F-measure_downbeat", "F-measure_avg"]]

### everything together

In [ ]:
df = pd.concat([final_df, final_df_99, final_df_hpo, final_df_4], axis = 0) 
df = remove_cemgil(df)
df

In [ ]:
df = pd.concat([final_df_99, final_df_hpo], axis = 0) # test set
df = remove_cemgil(df)
#columns =["F-measure", "AMLt","CMLt", "ACR_L2_any"]
columns =["F-measure", "AMLt","CMLt",]
cols = [col for col in df.columns if any(c in col for c in columns)]
df = df[cols]
df
for row in df_to_latex_rows(df, rename_idx=[  "Baseline", "Baselie (HPO)"], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col= True):
    print(row) 
df

In [70]:
df = pd.concat([final_df_99, final_df_hpo,  final_df_2, final_df_4, final_df_4_wt3], axis = 0) # test set
df = remove_cemgil(df)
columns =["F-measure", "AMLt","CMLt"]
#columns =["ACR_L2_any", "ACR_L3_any", "ACR_L4_any"]
cols = [col for col in df.columns if any(c in col for c in columns)]
df = df[cols]
df
for row in df_to_latex_rows(df, rename_idx=[  "Baseline", "Baseline (HPO)",  "2", "4", "4 (wt 3)",], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= True, bold_max_per_col= True):
    print(row) 
df

F-measure_beat
{'baseline 99': 92.71, 'baseline_hpo_trial_12': 92.85, 'clusters2_layer6': 92.92, 'clusters4_layer12': 92.81, 'clusters4_layer12 wt3': 92.82}
CMLt_beat
{'baseline 99': 84.86, 'baseline_hpo_trial_12': 85.28, 'clusters2_layer6': 85.43, 'clusters4_layer12': 85.13, 'clusters4_layer12 wt3': 85.18}
AMLt_beat
{'baseline 99': 88.73, 'baseline_hpo_trial_12': 89.09, 'clusters2_layer6': 89.3, 'clusters4_layer12': 88.99, 'clusters4_layer12 wt3': 89.04}
F-measure_downbeat
{'baseline 99': 85.34, 'baseline_hpo_trial_12': 85.45, 'clusters2_layer6': 85.53, 'clusters4_layer12': 85.69, 'clusters4_layer12 wt3': 85.69}
CMLt_downbeat
{'baseline 99': 72.85, 'baseline_hpo_trial_12': 72.94, 'clusters2_layer6': 73.19, 'clusters4_layer12': 73.48, 'clusters4_layer12 wt3': 73.48}
AMLt_downbeat
{'baseline 99': 80.37, 'baseline_hpo_trial_12': 80.18, 'clusters2_layer6': 80.43, 'clusters4_layer12': 80.66, 'clusters4_layer12 wt3': 80.66}
\textbf{Baseline} & $92.71 \pm 0.21$ & $84.86 \pm 0.38$ & $88.73 \p

,F-measure_beat,CMLt_beat,AMLt_beat,F-measure_downbeat,CMLt_downbeat,AMLt_downbeat
baseline 99,92.71 ± 0.21,84.86 ± 0.38,88.73 ± 0.35,85.34 ± 0.26,72.85 ± 0.45,80.37 ± 0.54
baseline_hpo_trial_12,92.85 ± 0.10,85.28 ± 0.24,89.09 ± 0.27,85.45 ± 0.21,72.94 ± 0.55,80.18 ± 0.61
clusters2_layer6,92.92 ± 0.17,85.43 ± 0.34,89.30 ± 0.21,85.53 ± 0.16,73.19 ± 0.35,80.43 ± 0.36
clusters4_layer12,92.81 ± 0.10,85.13 ± 0.19,88.99 ± 0.25,85.69 ± 0.26,73.48 ± 0.48,80.66 ± 0.28
clusters4_layer12 wt3,92.82 ± 0.09,85.18 ± 0.19,89.04 ± 0.23,85.69 ± 0.26,73.48 ± 0.51,80.66 ± 0.33


In [ ]:
df_means_only = final_df.apply(extract_mean, axis=1)
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

In [ ]:

df_means_only = final_df.apply(extract_mean, axis=1) # skipped 3
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline


In [ ]:
def extract_mean(series):
    return series.str.split("±").str[0].astype(float)
df_means_only = final_df.apply(extract_mean, axis=1) # skipped 1, 3
diff_vs_baseline = df_means_only.subtract(df_means_only.loc["baseline"])
diff_vs_baseline = diff_vs_baseline.drop(index="baseline")
diff_vs_baseline

## all data metrics

In [ ]:
def extract_mean_results(scores, decimals = 4, name = None, trial = None):
    res = extract_mean_std(scores, trial = trial, extract_std=True, decimals = decimals, name=name)
    res = pd.DataFrame(res)
    return res
def get_scores_for_plots(json_path, cluster, cluster_config, df_path, seeds, name, metric):
    if "baseline" in name:
        cluster_model = None
        cluster_config = None
    else:
        cluster_model = int(name.split("cluster_")[1][0]) 
    
    cluster_model_full_results, genres =get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = cluster_config, filter_files = False, name = name, return_per_track = False, subfolder = "all_data_metrics", cluster_model = cluster_model )
    cluster_model_cluster_results, genres = get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = cluster_config, filter_files = True, name = name, return_per_track = False, subfolder = "all_data_metrics" , cluster_model = cluster_model  )
    scores_full = extract_mean_results (cluster_model_full_results, decimals =4, name = f"data_full", trial = name)
    scores_cluster = extract_mean_results (cluster_model_cluster_results, decimals =4, name = f"data_cluster{cluster}", trial = name)
    df_full = pd.concat ([scores_full, scores_cluster],  axis=0)
    return df_full[[metric]]
def get_scores_from_table(df):
    index = df.index
    index_full = [idx for idx in index if "data_full" in idx]
    index_cluster = [idx for idx in index if "data_cluster" in idx]
    full_scores = df.loc[index_full].values
    cluster_scores = df.loc[index_cluster].values

    full_scores_mean = [float(score[0].split(" +-")[0]) for score in full_scores]
    full_scores_std = [float(score[0].split("+- ")[1]) for score in full_scores]
    cluster_scores_mean = [float(score[0].split(" +-")[0]) for score in cluster_scores]
    cluster_scores_std = [float(score[0].split("+- ")[1]) for score in cluster_scores]

    return (full_scores_mean, full_scores_std, cluster_scores_mean, cluster_scores_std)
def get_all_metrics_cluster(seeds, json_path,names, cluster, cluster_config, df_path, metric ):
    
    df_total = []
    for model_name in names:
        df_scores = get_scores_for_plots(json_path, cluster, cluster_config, df_path, seeds, model_name, metric=metric)
        print(f"Model: {model_name}")
        df_total.append(df_scores)
    df_total = pd.concat (df_total, axis=0)
    return df_total

In [ ]:
json_path = "json_test_scores"
cluster, cluster_config = 1, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
df = get_scores_for_plots(json_path, cluster, cluster_config, df_path, seeds, names[0], metric= "F-measure_downbeat")
df

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
json_path = "json_test_scores"
names =["baseline_hpo_trial_12_best","cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
cluster, cluster_config = 3, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
df_total = []
for model_name in names:
    df_scores = get_scores_for_plots(json_path, cluster, cluster_config, df_path, seeds, model_name, metric="F-measure_downbeat")
    print(f"Model: {model_name}")
    df_total.append(df_scores)
df_total = pd.concat (df_total, axis=0)
df_total

In [ ]:
names =["paper_baseline_99","cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"] #"baseline_hpo_trial_12_best"
cluster_config =  "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
type = "downbeat"
metric = f"F-measure_{type}"
seeds = [0,1,2,3,4,5,6,7,8,9]
dataframes = []
json_path = "json_test_scores"
cluster = 1
df_total = get_all_metrics_cluster(seeds, json_path,names, cluster, cluster_config, df_path, metric )
df_total

### 4 clusters 

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
dataframes = []
json_path = "json_test_scores"
names =["paper_baseline_99","cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"] #"baseline_hpo_trial_12_best"
cluster_config =  "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
type = "downbeat"
metric = f"F-measure_{type}"
#metric = "ACR_L4_any_downbeat"
for cluster in [1,2,3,4]:
    df_total = get_all_metrics_cluster(seeds, json_path,names, cluster, cluster_config, df_path, metric )
    dataframes.append(df_total)

In [ ]:
dataframes[0]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming you have a list of dataframes and corresponding cluster names
#dataframes = [df_total_cluster1, df_total_cluster2, df_total_cluster3, ...]  # Your dataframes
cluster_names = ['cluster 1', 'cluster 2', 'cluster 3', 'cluster 4']  # Your cluster names

models =names
colors = ['#3b82f6', '#ef4444', '#10b981', "#e60bc1", "#f6c901"]

# Calculate number of rows needed (2 columns)
n_clusters = len(dataframes)
n_rows = (n_clusters + 1) // 2  # Ceiling division

# Create subplots
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 5 * n_rows))
fig.suptitle(f'{metric}', fontsize=12, fontweight='bold')

# Flatten axes array for easier iteration
axes = axes.flatten() if n_clusters > 1 else [axes]

# Plot for each cluster
for idx, (df_total, cluster) in enumerate(zip(dataframes, cluster_names)):
    ax = axes[idx]
    
    # Get scores for this cluster
    full_scores_mean, full_scores_std, cluster_scores_mean, cluster_scores_std = get_scores_from_table(df_total)
    
    # Plot each model with error bars
    for i, model in enumerate(models):
        ax.errorbar(full_scores_mean[i], cluster_scores_mean[i], 
                    xerr=full_scores_std[i], yerr=cluster_scores_std[i],
                    fmt='o', markersize=8, capsize=4, capthick=2,
                    color=colors[i], label=model, linewidth=2)
    
    # Customize the subplot
    ax.set_xlabel('Full Data Result', fontsize=10, fontweight='bold')
    ax.set_ylabel('Cluster Data Result', fontsize=10, fontweight='bold')
    # ax.set_ylim(0, 100)
    # ax.set_xlim(0, 100)
    ax.set_title(f'{cluster}', fontsize=11, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide any unused subplots if odd number of clusters
if n_clusters % 2 != 0:
    axes[-1].set_visible(False)

# Adjust layout and display
plt.tight_layout()
plt.show()

### 2 clusters

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9]
dataframes = []
json_path = "json_test_scores"
names =["paper_baseline_99","cluster_1_trial_47", "cluster_2_trial_69"]
cluster_config =  "clusters2_layer6"
df_path="data_cluster_assignments/df_hard_6_2clusters_new.csv"
metric = "F-measure_beat"
for cluster in [1,2]:
    df_total = get_all_metrics_cluster(seeds, json_path,names, cluster, cluster_config, df_path, metric )
    dataframes.append(df_total)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming you have a list of dataframes and corresponding cluster names
#dataframes = [df_total_cluster1, df_total_cluster2, df_total_cluster3, ...]  # Your dataframes
cluster_names = ['cluster 1', 'cluster 2']  # Your cluster names

models =names
colors = ['#3b82f6', '#ef4444', '#10b981', "#e60bc1", "#f6c901"]

# Calculate number of rows needed (2 columns)
n_clusters = len(dataframes)
n_rows = (n_clusters + 1) // 2  # Ceiling division

# Create subplots
fig, axes = plt.subplots(n_rows, 2, figsize=(12, 5 * n_rows))
fig.suptitle(f'{metric}', fontsize=12, fontweight='bold')

# Flatten axes array for easier iteration
axes = axes.flatten() if n_clusters > 1 else [axes]

# Plot for each cluster
for idx, (df_total, cluster) in enumerate(zip(dataframes, cluster_names)):
    ax = axes[idx]
    
    # Get scores for this cluster
    full_scores_mean, full_scores_std, cluster_scores_mean, cluster_scores_std = get_scores_from_table(df_total)
    
    # Plot each model with error bars
    for i, model in enumerate(models):
        ax.errorbar(full_scores_mean[i], cluster_scores_mean[i], 
                    xerr=full_scores_std[i], yerr=cluster_scores_std[i],
                    fmt='o', markersize=8, capsize=4, capthick=2,
                    color=colors[i], label=model, linewidth=2)
    
    # Customize the subplot
    ax.set_xlabel('Full Data Result', fontsize=10, fontweight='bold')
    ax.set_ylabel('Cluster Data Result', fontsize=10, fontweight='bold')
    # ax.set_ylim(0, 100)
    # ax.set_xlim(0, 100)
    ax.set_title(f'{cluster}', fontsize=11, fontweight='bold')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

# Hide any unused subplots if odd number of clusters
if n_clusters % 2 != 0:
    axes[-1].set_visible(False)

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
seeds = [0,1,2,3,4,5,6,7,8,9] #1, 2,3,4
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
json_path = "json_test_scores"
# df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
#                                                                        full = True, names = names, decimals=4 ) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters4_layer4 = get_total_results_new(seeds = seeds, json_val_path = json_path, 
         layer = 12, clusters = 4, names = names,  
         skip_cluster = [3], df_path="data_cluster_assignments/df_cmeans_12_4clusters_new.csv", decimals=4,
          subfolders=["all_data_metrics", "all_data_metrics"] )
configs = {
    "clusters4_layer12": df_final_clusters4_layer4,
    
    #"baseline": df_final_baseline
}
final_df_4 = pd.DataFrame(configs).T 
# #final_df["F1average"] = final_df[["F-measure_beat","F-measure_downbeat"]].mean(axis=1).round(3).astype(str)
final_df_4 = remove_cemgil(final_df_4)
#diff_vs_baseline = get_baseliine_diff (final_df , num_trials=1)
final_df_4


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data
full_scores_mean, full_scores_std, cluster_scores_mean, cluster_scores_std = get_scores_from_table(df_total)
models =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]

full_data = full_scores_mean
full_errors = full_scores_std
cluster_data = cluster_scores_mean
cluster_errors = cluster_scores_std
# Create the scatter plot
plt.figure(figsize=(5, 5))

# Plot each model with error bars
colors = ['#3b82f6', '#ef4444', '#10b981', "#e60bc1", "#f6c901" ]
for i, model in enumerate(models):
    plt.errorbar(full_data[i], cluster_data[i], 
                 xerr=full_errors[i], yerr=cluster_errors[i],
                 fmt='o', markersize=10, capsize=5, capthick=2,
                 color=colors[i], label=model, linewidth=2)

# Customize the plot
plt.xlabel('Full Data Result', fontsize=12, fontweight='bold')
plt.ylabel('Cluster Data Result', fontsize=12, fontweight='bold')
plt.title(f'Full Data vs Cluster Data Performance for cluster {cluster}', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
# plt.ylim(75, 85)
# plt.xlim(75, 85)
# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
json_path = "json_test_scores"
cluster, cluster_config = 1, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
seeds = [0,1,2,3,4,5,6,7,8,9]
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
df = get_scores_for_plots(json_path, cluster, cluster_config, df_path, seeds, names[2], metric= "F-measure_downbeat")
df

In [ ]:
json_path = "json_test_scores"
cluster, cluster_config = 1, "0clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
seeds = [0,1,2,3,4]
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
cluster_model_full_results, genres =get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = cluster_config, filter_files = False, name = names[1], return_per_track = False, subfolder = "all_data_metrics" )
cluster_model_cluster_results, genres = get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = cluster_config, filter_files = True, name = names[1], return_per_track = False, subfolder = "all_data_metrics" )
scores_full = extract_mean_results (cluster_model_full_results, decimals =4, name = f"data_full", trial = name)
scores_cluster = extract_mean_results (cluster_model_cluster_results, decimals =4, name = f"data_cluster{cluster}", trial = name)
metric=["F-measure_downbeat"]
df_full = pd.concat ([scores_full, scores_cluster],  axis=0)
df_full[metric]

In [ ]:

json_file = open("json_test_scores/all_data_metrics/clusters4_layer12/1/cluster_1_trial_10/epoch_best_seed_7_cluster_1_trial_10S_SEED_shift_tolerant_weighted_bce-h512-augTrueTrueTrue.json")
        
json1_str = json_file.read()
json1_data = json.loads(json1_str)
print(len(json1_data))
json_data = {key[:-10]: value for key, value in json1_data.items()} #{key[:-10]: value for key, value in json1_data.items()}
df  = pd.DataFrame(json_data).T
#columns = [col for col in df.columns.values if "L3" in col]
#columns = [col for col in df.columns.values if "L3" not in col and "L4" not in col ]
columns = ['F-measure_beat', 'Cemgil_beat', 'CMLt_beat', 'AMLt_beat', 'F-measure_downbeat', 'Cemgil_downbeat', 'CMLt_downbeat', 'AMLt_downbeat', 'ACR_L2_any_beat', "ACR_L4_any_downbeat"]
#df[columns].mean()
df[columns].mean()

In [ ]:
json_file = open("/home/ui556004/projects/beat_this/json_test_scores/all_data_metrics/paper_baseline_best/epoch_best_seed_2_full_data_intermediate_checkpointsS_SEED_shift_tolerant_weighted_bce-h512-augTrueTrueTrue.json")
        
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json_data = {key[:-10]: value for key, value in json1_data.items()} #{key[:-10]: value for key, value in json1_data.items()}
df  = pd.DataFrame(json_data).T
#columns = [col for col in df.columns.values if "L3" in col]
#columns = [col for col in df.columns.values if "L3" not in col and "L4" not in col ]
columns = ['F-measure_beat', 'Cemgil_beat', 'CMLt_beat', 'AMLt_beat', 'F-measure_downbeat', 'Cemgil_downbeat', 'CMLt_downbeat', 'AMLt_downbeat', 'ACR_L2_any_beat', "ACR_L4_any_downbeat"]
#df[columns].mean()
df[columns].mean()

In [ ]:
json_file = open("json_test_scores/all_data_metrics/paper_baseline_99/0epoch_100_seed_0_periodic.json")
        
json1_str = json_file.read()
json1_data = json.loads(json1_str)
json_data = {key[:-10]: value for key, value in json1_data.items()} #{key[:-10]: value for key, value in json1_data.items()}
df  = pd.DataFrame(json_data).T
#columns = [col for col in df.columns.values if "L3" in col]
#columns = [col for col in df.columns.values if "L3" not in col and "L4" not in col ]
columns = ['F-measure_beat', 'Cemgil_beat', 'CMLt_beat', 'AMLt_beat', 'F-measure_downbeat', 'Cemgil_downbeat', 'CMLt_downbeat', 'AMLt_downbeat', 'ACR_L2_any_beat', "ACR_L4_any_downbeat"]
#df[columns].mean()
df[columns].mean()

## hypothesis testing


In [ ]:
def per_cluster_average(seeds, name, cluster, metric,  cluster_config,df_path, json_path= "json_test_scores", filter_files = True ):
    cluster_dataframes = []
    #filter_files = False if "baseline" in name else True
    for seed in seeds:
        seed = [seed]
        cluster_model_cluster_results = get_restults_per_cluster (json_path,  cluster, seed, df_path, full= False,cluster_config = cluster_config, filter_files = filter_files, name = name, return_per_track = True, subfolder = "all_data_metrics"  )
        cluster_dataframes.append(cluster_model_cluster_results[[metric]])
    df_mean = pd.concat(cluster_dataframes).groupby(level=0).mean()
    indices = df_mean.index
    if "track" in indices[0]:
        df_mean.index = df_mean.index.str.replace("/track.npy", "", regex=False)
    return df_mean

In [ ]:
json_path = "json_test_scores"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
metric = "F-measure_beat"
seeds = [0,1,2,3,4,5,6,7,8,9]

df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
df_full= per_cluster_average(seeds = seeds, name = "paper_baseline_99", cluster= cluster, metric= metric,df_path= df_path, cluster_config= None)
df_full = df_full.sort_index()
df_full.mean()

In [ ]:
from scipy.stats import wilcoxon
json_path = "json_test_scores"
cluster, cluster_config = 4, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
metric = "F-measure_downbeat"
names =["paper_baseline_99", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
seeds = [0,1,2,3,4,5,6,7,8,9]
df_total = []
skip_cluster = [1]
for cluster in [1,2,3,4]:
    if cluster in skip_cluster:
        print(f"Skipping cluster {cluster}")
        df_cluster = df_full= per_cluster_average(seeds = seeds, name = names[0], cluster= cluster, metric= metric,df_path= df_path, cluster_config= None, filter_files = True)
    else:
        df_cluster = per_cluster_average(seeds = seeds, name = names[cluster], cluster= cluster, metric= metric, df_path= df_path, cluster_config= cluster_config, filter_files = True)
    df_total.append(df_cluster)
df_total = pd.concat (df_total, axis=0)
df_full= per_cluster_average(seeds = seeds, name = names[0], cluster= cluster, metric= metric,df_path= df_path, cluster_config= None, filter_files = False)
assert len(df_total) == 993
assert(len(df_full) == 993)
df_total = df_total.sort_index()
print(f"mean of clustered: {df_total.mean()}")
print(f"mean of df_full: {df_full.mean()}")
difference = df_total - df_full
print("Difference statistics (clustered - full data):")
print(difference.mean())
print(difference.std())
#print((difference > 0).mean())
stat, p = wilcoxon(difference, alternative="greater")
print(f"Wilcoxon test statistic: {stat}, p-value: {p}")

In [ ]:
json_path = "json_test_scores"
cluster, cluster_config = 2, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
metric = "F-measure_beat"
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
name = names[2]
seeds = [0,1,2,3,4,5,6,7,8,9]
cluster_dataframes = []
for seed in seeds:
    seed = [seed]
    cluster_model_cluster_results = get_restults_per_cluster (json_path,  cluster, seed, df_path, full= False,cluster_config = cluster_config, filter_files = True, name = name, return_per_track = True, subfolder = "all_data_metrics"  )
    cluster_dataframes.append(cluster_model_cluster_results[[metric]])
df_mean = pd.concat(cluster_dataframes).groupby(level=0).mean()
df_mean

In [ ]:
json_path = "json_test_scores"
cluster, cluster_config = 2, "clusters4_layer12"
df_path= "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
seeds = [7]
names =["baseline_hpo_trial_12_best", "cluster_1_trial_10", "cluster_2_trial_16", "cluster_3_trial_72", "cluster_4_trial_43"]
cluster_model_cluster_results = get_restults_per_cluster (json_path,  cluster, seeds, df_path, full= False,cluster_config = cluster_config, filter_files = True, name = names[2], return_per_track = True, subfolder = "all_data_metrics"  )
cluster_model_cluster_results[[metric]]

# tables

In [ ]:
def get_table(json_path, seeds, df_path, num_clusters, 
              layer, trial_per_cluster, name_full = "full_model_trial12", decimals = 2, skip_clusters = [], bold_difference=True, bold_baseline_if_best= True):
    rows = []
    configs = {}
    cluster_config = f"clusters{num_clusters}_layer{layer}"
    for cluster in range(1, num_clusters+1):

        df_total_cluster = get_per_cluster_results(trials = [trial_per_cluster[cluster] ], seeds = seeds,  df_path = df_path,
                                        json_path = json_path, name_full = name_full, cluster = cluster, cluster_config=cluster_config,
                                        extrat_std=True, decimals=decimals)
        rows_cluster = df_to_latex_rows(df_total_cluster, rename_idx=[f"Cluster {cluster}", f"Baseline {cluster}"],bold_difference= bold_difference, bold_baseline_if_best= bold_baseline_if_best)
        print(rows_cluster)
        rows += rows_cluster
# all data
    names = ["full_model_trial12"]+[f"cluster_{i}_trial_{trial_per_cluster[i]}" for i in range(1, num_clusters+1)]
    print(names)
    df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
                                                                        full = True, names = names, decimals = decimals) #rename_columns="baseline"
    df_seeds_mean, df_seeds_std, df_final_clusters = get_total_results_new(seeds = seeds, json_val_path = json_path, skip_cluster=[],
                                                                                layer = layer, clusters = num_clusters , names = names, df_path=df_path, decimals = decimals )
    rename_idx = ["Total (clusters)", "Total (baseline)"]
    configs[f"clusters{num_clusters}_layer{layer}"] = df_final_clusters

    if len(skip_clusters) > 0:
        df_seeds_mean, df_seeds_std, df_final_clusters_skipped = get_total_results_new(seeds = seeds, json_val_path = json_path, skip_cluster=skip_clusters,
                                                                                layer = layer, clusters = num_clusters , names = names, df_path=df_path, decimals = decimals )
        rename_idx.insert(1, f"Total (clusters wt {','.join(str(x) for x in skip_clusters)})") 
        configs[ f"clusters{num_clusters}_layer{layer}_skipped"] = df_final_clusters_skipped
    configs["baseline"] = df_final_baseline
    final_df = pd.DataFrame(configs).T 
    final_df = remove_cemgil(final_df)
    for r in rows:
        print(r)
    print("\midrule")
    for row in df_to_latex_rows(final_df, rename_idx=rename_idx, bold_index = True, pm= True, bold_difference= bold_difference, bold_baseline_if_best= bold_baseline_if_best):
        print(row) 

In [ ]:
df
for row in df_to_latex_rows(df, rename_idx=[f"Baseline best", "Baseline 99", "Baseline HPO", "Clusters 4 layer 12", "Clusters 2 layer 6"], bold_index = True, pm= True, bold_difference=False, bold_baseline_if_best= False, bold_max_per_col= True):
    print(row) 

In [ ]:
row = ['\\textbf{Cluster 2} & $79.05 \\pm 0.25$ & $65.39 \\pm 0.99$ & $79.40 \\pm 0.85$ & $64.20 \\pm 0.51$ & $48.70 \\pm 1.45$ & $62.80 \\pm 1.19$ \\\\', '\\textbf{Baseline 2} & $79.58 \\pm 0.24$ & $66.27 \\pm 0.69$ & $79.85 \\pm 0.65$ & $64.20 \\pm 0.75$ & $47.62 \\pm 1.29$ & $62.46 \\pm 0.80$ \\\\']
cluster = row[0]
beat_f1 = cluster


In [ ]:
json_path = "json_val_scores"
seeds = [0,1,2,3,4,5,6,7,8,9]
df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv"
name_full = "full_model_trial12"
num_clusters, layer  = 2, 6
trial_per_cluster = {1: 47, 2:69}
get_table(json_path, seeds, df_path, num_clusters, layer, trial_per_cluster)

In [ ]:
json_path = "json_test_scores"
seeds = [0,1,2,3,4,5,6,7,8,9]
df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv"
name_full = "full_model_trial12"
num_clusters, layer  = 2, 6
trial_per_cluster = {1: 47, 2:69}
get_table(json_path, seeds, df_path, num_clusters, layer, trial_per_cluster)

In [ ]:
json_path = "json_test_scores"
seeds = [0,1,2,3,4,5,6,7,8,9]
df_path = "data_cluster_assignments/df_cmeans_12_4clusters_new.csv"
name_full = "full_model_trial12"
num_clusters, layer  = 4, 12
trial_per_cluster = {1: 10, 2:16, 3: 72, 4: 43}
get_table(json_path, seeds, df_path, num_clusters, layer, trial_per_cluster, skip_clusters=[3])

In [ ]:
import math

def sci_latex(value, decimals=3):
    """Convert float to a.bcd * 10^{k} latex format"""
    if value == 0:
        return "0"
    exp = int(math.floor(math.log10(abs(value))))
    mant = value / (10 ** exp)
    return rf"{mant:.{decimals}f} \times 10^{{{exp}}}"

In [ ]:
def get_params_table(num_clusters, storage_path,trial_per_cluster, decimals = 3):
    rows = []
    for cluster in range(1, num_clusters + 1):
        study_name = f"{num_clusters}clusters_cluster{cluster}" if num_clusters ==2 else f"{num_clusters}clusters_cluster_{cluster}"
       # study_name = f"{num_clusters}clusters_cluster{cluster}"
        study = optuna.load_study(
            study_name=study_name,
            storage=storage_path
        )
        
        params = study.trials[trial_per_cluster[cluster]].params
        
        lr = sci_latex(params["lr"], decimals)
        weight_decay = f"{params['weight_decay']:.4f}"
        batch = params["batch_size"]
        freeze = params["freeze_layers"]
        checkpoint = params["checkpoint"]

        row = (
            rf"\textbf{{Cluster {cluster}}} & "
            f"{batch} & "
            f"${lr}$ & "
            f"{weight_decay} & "
            f"{freeze} & "
            f"{checkpoint} \\\\"
        )
        
        rows.append(row)

    for r in rows:
        print(r)

In [ ]:
import optuna
trial_per_cluster = {1: 47, 2:69}
num_clusters = 2
storage_path = "sqlite:///optuna_new.db"
get_params_table(num_clusters, storage_path,trial_per_cluster, decimals = 3)

In [ ]:
import optuna
trial_per_cluster = {1: 10, 2:16, 3: 72, 4: 43}
num_clusters = 4
storage_path = "sqlite:///optuna_down.db"
get_params_table(num_clusters, storage_path,trial_per_cluster, decimals = 3)

In [ ]:
import optuna
trial_per_cluster = {1: 47, 2:69}
num_clusters = 2
storage_path = "sqlite:///optuna_new.db"
for cluster in range (1, num_clusters +1):
    storage_path = f"{num_clusters}clusters_cluster{cluster}"
    study_name = f"{num_clusters}clusters_cluster{cluster}"
    study = optuna.load_study(
        study_name=study_name,
        storage="sqlite:///optuna_new.db"
    )
    print(study.trials[trial_per_cluster[cluster]].params)
    #print(study.study_name)

   

In [ ]:
study_name

In [ ]:
json_path = "json_val_scores"
seeds = [0,1,2,3,4,5,6,7,8,9]
df_path = "data_cluster_assignments/df_hard_6_2clusters_new.csv"
name_full = "full_model_trial12"
num_clusters, layer  = 2, 6
rows = []
trial_per_cluster = {1: 47, 2:69}
decimals = 2
for cluster in range(1, num_clusters+1):

    df_total_cluster = get_per_cluster_results(trials = [trial_per_cluster[cluster] ], seeds = seeds,  df_path = df_path,
                                    json_path = json_path, name_full = name_full, cluster = cluster, 
                                    extrat_std=True, decimals=decimals)
    rows_cluster = df_to_latex_rows(df_total_cluster, rename_idx=[f"Cluster {cluster}", f"Baseline {cluster}"])
    rows += rows_cluster
# all data
names = ["full_model_trial12"]+[f"cluster_{i}_trial_{trial_per_cluster[i]}" for i in range(1, num_clusters+1)]
print(names)
df_seeds_mean, df_seeds_std, df_final_baseline = get_total_results_new(seeds = seeds, json_val_path = json_path, 
                                                                       full = True, names = names, decimals = decimals) #rename_columns="baseline"
df_seeds_mean, df_seeds_std, df_final_clusters = get_total_results_new(seeds = seeds, json_val_path = json_path, skip_cluster=[],
                                                                               layer = layer, clusters = num_clusters , names = names, df_path=df_path, decimals = decimals )
configs = {
    f"clusters{num_clusters}_layer{layer}": df_final_clusters,
    
    "baseline": df_final_baseline
}
final_df = pd.DataFrame(configs).T 
final_df = remove_cemgil(final_df)
for r in rows:
    print(r)
for row in df_to_latex_rows(final_df, rename_idx=["Total (clusters)", "Total (baseline)"], bold_index = True):
    print(row)    #df_total

In [ ]:
rows_cluster

In [ ]:
df_total_cluster